# Smile Dynamics II — Bergomi (2005)
## Implémentation complète : Modèle de Variance Forward sur SX5E

---

> **Référence :** Lorenzo Bergomi, *Smile Dynamics II*, Société Générale, March 2005 (SSRN 1493302).  
> **Continuation directe de :** Bergomi (2004) — *Smile Dynamics I*

---

## Motivation

Dans **Smile Dynamics I**, Bergomi a montré que les modèles traditionnels (Heston, Jump/Lévy) imposent des **contraintes structurelles rigides** entre :
1. La dynamique des vols implicites (vol-de-vol)
2. Le forward skew
3. La corrélation spot/vol

**Smile Dynamics II** propose un modèle qui **découple** ces trois ingrédients, en modélisant directement la dynamique de la courbe des **Forward Variance Swaps (FVS)**, notées $\xi^T_t$.

---

## Plan du notebook

| Section | Contenu |
|---|---|
| **0** | Setup & données (MDX / SX5E) |
| **1** | Modélisation des Variance Swaps Forward |
| **2** | Modèle à 1 facteur — processus OU |
| **3** | Modèle à 2 facteurs — facteurs court & long |
| **4** | Modèle à N facteurs — structure de corrélation |
| **5** | Spécification du processus spot (CEV discret) |
| **6** | Équation de pricing & algorithme Monte Carlo |
| **7** | Structure par terme de la vol-de-vol |
| **8** | Structure par terme du skew — décomposition intrinsèque/correlation |
| **9** | Pricing d'options exotiques (Reverse Cliquet, Napoleon, Accumulator) |
| **10** | Options sur variance réalisée |
| **11** | Calibration sur données SX5E |
| **12** | Synthèse & comparaison Bergomi I vs II |

---
## Section 0 — Setup & données de marché

In [ ]:
# ============================================================
#  IMPORTS
# ============================================================
import datetime as dt
import warnings
import pickle
from pathlib import Path
from itertools import product as iproduct

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import PercentFormatter
from scipy.stats import norm
from scipy.optimize import brentq, minimize
from scipy.integrate import quad
from pandas.tseries.offsets import BDay

warnings.simplefilter('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# Seed pour reproductibilité
RNG_SEED = 42
print('Imports OK')

In [ ]:
# ============================================================
#  CONNEXION MDX
# ============================================================
import ezmdx
from maxxpy.apis.mdx.api import MdxClient

LOGIN_MDX    = ""   # <-- TON LOGIN
PASSWORD_MDX = ""   # <-- TON MOT DE PASSE

MDX_TYPES = {
    'volatility': 'EQUITY_VOLATILITY',
    'spot': ['STOCK_QUOTE', 'INDEX_QUOTE', 'FUND_QUOTE']
}
EUROSTOXX_MDX_CODE = 'STOX5E_X'

ezmdx.set_app(app_name='VEGA5')
ezmdx.prod.satis_login()
mtx_client = MdxClient('MSD', LOGIN_MDX, PASSWORD_MDX, use_prod_only=True)

In [ ]:
# ============================================================
#  PLAGE DE DATES & CHARGEMENT DES DONNÉES
# ============================================================
today      = pd.Timestamp.today().normalize()
date_end   = today - BDay(1)
date_start = date_end - pd.DateOffset(years=5)

print(f'Période : {date_start.date()}  →  {date_end.date()}')

cache_dir  = Path('./cache')
cache_dir.mkdir(exist_ok=True)
cache_path = cache_dir / 'sx5e_bergomi_5y_cache.pkl'

def get_market_data(mtx_client, asset_name, date_range, mdx_type):
    return mtx_client.get_market_data(mdx_type=mdx_type, code=asset_name, date=date_range)

def get_vol(asset_name, asset_type, date_start, date_end, mtx_client):
    bdays = pd.bdate_range(date_start, date_end).strftime('%Y-%m-%d').tolist()
    df = get_market_data(mtx_client, f'{asset_type}_{asset_name}', bdays, MDX_TYPES['volatility'])
    return df[['STRIKE', 'MATURITY', 'VOLATILITY', 'DATE']].copy()

def get_spot(mtx_client, asset_name, date_start, date_end):
    bdays = pd.bdate_range(date_start, date_end).strftime('%Y-%m-%d').tolist()
    for mtype in MDX_TYPES['spot']:
        try:
            return get_market_data(mtx_client, asset_name, bdays, mtype)
        except Exception:
            continue

if cache_path.exists():
    with open(cache_path, 'rb') as f:
        cached = pickle.load(f)
    vols_raw, spots_raw = cached['vols'], cached['spots']
    print('Cache chargé.')
else:
    vols_raw  = get_vol(EUROSTOXX_MDX_CODE, 'I', date_start, date_end, mtx_client)
    spots_raw = get_spot(mtx_client, EUROSTOXX_MDX_CODE, date_start - BDay(5), date_end + BDay(5))
    with open(cache_path, 'wb') as f:
        pickle.dump({'vols': vols_raw, 'spots': spots_raw}, f)
    print('Données fetchées et mises en cache.')

In [ ]:
# ============================================================
#  FONCTIONS UTILITAIRES BS & CONSTRUCTION SURFACE
# ============================================================
def bs_price(S, K, T, sigma, r=0., q=0., option='call'):
    F  = S * np.exp((r - q) * T)
    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    disc = np.exp(-r * T)
    if option == 'call':
        return disc * (F * norm.cdf(d1) - K * norm.cdf(d2))
    return disc * (K * norm.cdf(-d2) - F * norm.cdf(-d1))


def implied_vol(S, K, T, price, r=0., q=0., option='call'):
    intrinsic = max(S * np.exp(-q*T) - K * np.exp(-r*T), 0.)
    if price <= intrinsic + 1e-10:
        return np.nan
    try:
        return brentq(lambda v: bs_price(S, K, T, v, r, q, option) - price,
                      1e-4, 5.0, xtol=1e-8)
    except Exception:
        return np.nan


def build_surface(vols_raw, spots_raw, r=0., q=0.):
    vols = vols_raw.rename(columns={
        'STRIKE':'strike','MATURITY':'maturity_date',
        'VOLATILITY':'market_iv','DATE':'date'})
    vols['date']          = pd.to_datetime(vols['date'])
    vols['maturity_date'] = pd.to_datetime(vols['maturity_date'])
    vols['strike']        = pd.to_numeric(vols['strike'],    errors='coerce')
    vols['market_iv']     = pd.to_numeric(vols['market_iv'], errors='coerce')
    if vols['market_iv'].median() > 2:
        vols['market_iv'] /= 100.
    spots = spots_raw.copy()
    spots['date'] = pd.to_datetime(spots['DATE'])
    sc = [c for c in spots.columns if c != 'DATE'][0]
    spots['spot'] = pd.to_numeric(spots[sc], errors='coerce')
    df = vols.merge(spots[['date','spot']], on='date', how='left').dropna()
    df['bdays'] = [len(pd.bdate_range(d, m)) - 1
                   for d, m in zip(df['date'], df['maturity_date'])]
    df = df[df['bdays'] > 0]
    df['T']       = df['bdays'] / 252.
    df['forward'] = df['spot'] * np.exp((r - q) * df['T'])
    df['log_m']   = np.log(df['strike'] / df['forward'])
    return df.sort_values(['date','T','strike']).reset_index(drop=True)


surface = build_surface(vols_raw, spots_raw)
print(f'Surface : {surface.shape[0]:,} points — {surface["date"].nunique()} dates')
surface.head(8)

---
## Section 1 — Modélisation des Variance Swaps Forward

### 1.1 Définition & dérive nulle

Un **Variance Swap** (VS) de maturité $T$ paie à maturité :
$$V^h_{tT} - V^T_t$$
où $V^h_{tT}$ est la variance réalisée annualisée sur $[t,T]$ et $V^T_t$ la variance swap implicite.

On définit la **Forward Variance (FV)** instantanée comme :
$$\xi^T_t = V^{T,T}_t \quad \text{(variance forward pour la date } T \text{, vue de } t\text{)}$$

La VS variance de maturité $T$ s'écrit :
$$V^T_t = \frac{1}{T-t}\int_t^T \xi^u_t\,du$$

**Propriété fondamentale :** Bergomi montre par un argument d'arbitrage que la **dérive de pricing de toute FV est nulle** :

$$\boxed{d\xi^T_t = \xi^T_t \cdot \omega(T-t)\,dU_t}$$

La démonstration repose sur la construction d'un portefeuille auto-finançant long $\frac{T_2 - t}{T_2 - T_1}$ VS de maturité $T_2$ et short $\frac{T_1-t}{T_2-T_1}$ VS de maturité $T_1$, dont le P&L est exactement $V^{T_1,T_2}_{t+dt} - V^{T_1,T_2}_t$ au premier ordre en $dt$.

### 1.2 Courbe de Variance Forward observée sur le marché

La courbe $T \mapsto \xi^T_t$ est directement observable depuis la surface de volatilité implicite :
$$\xi^T_t = \frac{\partial}{\partial T}\left[(T-t)\hat{\sigma}^2_{VS}(T)\right]$$

In [ ]:
# ============================================================
#  EXTRACTION DE LA COURBE ξ^T_t DEPUIS LES DONNÉES DE MARCHÉ
#  via la Log Swap vol (réplication statique des VS)
# ============================================================
def logswap_vol(strikes, ivs, S, T, r=0., q=0.):
    """σ²_LS * T = 2 * ∫ OTM(K)/K² dK  (réplication statique VS)"""
    F    = S * np.exp((r - q) * T)
    mask = np.isfinite(ivs) & (ivs > 0)
    K_a, iv_a = strikes[mask], ivs[mask]
    if len(K_a) < 3:
        return np.nan
    px = np.array([
        bs_price(S, K, T, iv, r, q, 'put' if K <= F else 'call')
        for K, iv in zip(K_a, iv_a)
    ])
    return np.sqrt(max(2 * np.trapz(px / K_a**2, K_a) / T, 0))


def extract_vs_curve(surface, date):
    """
    Retourne la courbe T -> σ²_VS(T) pour une date donnée.
    """
    day_df = surface[surface['date'] == pd.Timestamp(date)]
    rows = []
    for T, grp in day_df.groupby('T'):
        sl = grp.sort_values('strike')
        S  = sl['spot'].iloc[0]
        lsv = logswap_vol(sl['strike'].values, sl['market_iv'].values, S, T)
        if np.isfinite(lsv):
            rows.append({'T': T, 'vs_vol': lsv, 'vs_var': lsv**2})
    return pd.DataFrame(rows).sort_values('T')


def extract_xi_curve(vs_curve):
    """
    Calcule ξ^T = d/dT[(T * σ²_VS(T))] par différences finies.
    """
    T   = vs_curve['T'].values
    var = vs_curve['vs_var'].values
    total_var = T * var  # T * σ²_VS
    # différence finie centrée
    xi = np.gradient(total_var, T)
    return pd.DataFrame({'T': T, 'xi': xi, 'vs_var': var})


# Exemple sur la date la plus récente disponible
latest_date = surface['date'].max()
vs_curve    = extract_vs_curve(surface, latest_date)
xi_curve    = extract_xi_curve(vs_curve)

print(f'Courbe VS extraite le {latest_date.date()} — {len(vs_curve)} maturités')
xi_curve.round(6)

In [ ]:
# ============================================================
#  FIGURE 1 — Courbe VS & Forward Variance (ξ^T)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gauche : Structure par terme σ_VS
axes[0].plot(vs_curve['T'] * 12, vs_curve['vs_vol'] * 100,
             'o-', color='steelblue', lw=2, ms=6)
axes[0].set_xlabel('Maturité (mois)')
axes[0].set_ylabel('σ_VS (%)')
axes[0].set_title(f'Structure par terme des Variance Swaps\n{latest_date.date()}')

# Droite : Courbe des Forward Variances ξ^T
axes[1].plot(xi_curve['T'] * 12, np.sqrt(np.maximum(xi_curve['xi'], 0)) * 100,
             'o-', color='firebrick', lw=2, ms=6)
axes[1].axhline(vs_curve['vs_vol'].iloc[0] * 100, ls='--', color='gray',
                label='σ_VS court terme')
axes[1].set_xlabel('Maturité T (mois)')
axes[1].set_ylabel('√ξ^T (%)')
axes[1].set_title(f'Courbe des Forward Variances ξ^T\n(dérivée de T·σ²_VS)')
axes[1].legend()

plt.suptitle('Section 1 — Courbe de Forward Variance (SX5E)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 2 — Modèle à 1 facteur

### 2.1 Spécification

On choisit une volatilité **exponentiellement décroissante** :
$$d\xi^T = \omega\,e^{-k_1(T-t)}\,\xi^T\,dU_t$$

Ce choix rend le modèle **markovien** : toutes les $\xi^T$ sont des fonctions d'un **unique facteur gaussien** $X_t$ :

$$\boxed{\xi^T(t) = \xi^T(0)\,\exp\!\left(\omega\,e^{-k_1(T-t)}X_t - \frac{\omega^2}{2}\,e^{-2k_1(T-t)}\mathbb{E}[X_t^2]\right)}$$

où $X_t$ est un processus **Ornstein-Uhlenbeck** :
$$dX_t = -k_1 X_t\,dt + dU_t, \qquad X_0 = 0$$

### 2.2 Récursions pour la simulation

Sur un pas $\delta$ :
$$X_{t+\delta} = e^{-k_1\delta}X_t + x_\delta, \qquad x_\delta \sim \mathcal{N}\!\left(0,\frac{1-e^{-2k_1\delta}}{2k_1}\right)$$
$$\mathbb{E}[X^2_{t+\delta}] = e^{-2k_1\delta}\mathbb{E}[X^2_t] + \frac{1-e^{-2k_1\delta}}{2k_1}$$

In [ ]:
# ============================================================
#  MODÈLE À 1 FACTEUR — CLASSE COMPLÈTE
# ============================================================
class BergomiOneFactor:
    """
    Modèle de Forward Variance à 1 facteur (Bergomi 2005, section 2.1).

    dξ^T = ω * e^{-k1*(T-t)} * ξ^T * dU_t

    Paramètres
    ----------
    omega : float  — vol-de-vol de la forward variance
    k1    : float  — vitesse de mean-reversion
    """
    def __init__(self, omega: float, k1: float):
        self.omega = omega
        self.k1    = k1

    def var_x(self, t: float) -> float:
        """E[X²_t] = (1 - e^{-2k1*t}) / (2*k1)"""
        return (1 - np.exp(-2 * self.k1 * t)) / (2 * self.k1)

    def xi(self, T: float, t: float, Xt: float,
            xi0: float, EX2: float) -> float:
        """
        Forward variance ξ^T(t) connaissant X_t et E[X²_t].
        Équation (2.1) du papier.
        """
        tau = T - t
        return xi0 * np.exp(
            self.omega * np.exp(-self.k1 * tau) * Xt
            - 0.5 * self.omega**2 * np.exp(-2 * self.k1 * tau) * EX2
        )

    def step_X(self, Xt: float, EX2: float,
                delta: float, rng: np.random.Generator):
        """Un pas de simulation Euler-exact pour X_t."""
        var_inc = (1 - np.exp(-2 * self.k1 * delta)) / (2 * self.k1)
        x_delta = rng.standard_normal() * np.sqrt(var_inc)
        X_new   = np.exp(-self.k1 * delta) * Xt + x_delta
        EX2_new = (np.exp(-2 * self.k1 * delta) * EX2 + var_inc)
        return X_new, EX2_new

    def vol_of_vs_vol(self, tau: float, dt: float = 1/12) -> float:
        """
        Volatilité annualisée de la VS vol pour maturité τ,
        mesurée sur un horizon dt.
        Vol(ln σ_VS(τ)) = ω * e^{-k1*τ} * sqrt((1 - e^{-2k1*dt}) / (2k1)) / sqrt(dt)
        """
        var_dx = (1 - np.exp(-2 * self.k1 * dt)) / (2 * self.k1)
        return self.omega * np.exp(-self.k1 * tau) * np.sqrt(var_dx / dt)


# Paramètres de référence Bergomi II
OMEGA_REF = 2.827
K1_REF    = 6.0    # 2 mois
K2_REF    = 0.25   # 4 ans
THETA_REF = 0.30   # poids facteur long
RHO_REF   = 0.0    # corrélation entre U et W

m1f = BergomiOneFactor(OMEGA_REF, K1_REF)

# Structure par terme de la vol-de-vol (1 facteur)
taus_months = np.arange(1, 61)
volvol_1f   = [m1f.vol_of_vs_vol(t/12, dt=1/12) for t in taus_months]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(taus_months, np.array(volvol_1f) * 100, 'steelblue', lw=2,
        label=f'1 facteur (ω={OMEGA_REF}, k₁={K1_REF})')
ax.set_xlabel('Maturité de la VS vol (mois)')
ax.set_ylabel('Vol de la VS vol (%)')
ax.set_title('Modèle 1 facteur — Structure par terme de la Vol-de-Vol')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Vol VS 1M (court terme) : {volvol_1f[0]*100:.1f}%')
print(f'Vol VS 1Y              : {volvol_1f[11]*100:.1f}%')
print(f'Vol VS 5Y              : {volvol_1f[59]*100:.1f}%')

---
## Section 3 — Modèle à 2 facteurs

### 3.1 Spécification (équation 2.2)

Pour une plus grande flexibilité de la structure par terme, on utilise **deux facteurs** :

$$d\xi^T = \omega\,\xi^T\!\left(e^{-k_1(T-t)}\,dU_t + \theta\,e^{-k_2(T-t)}\,dW_t\right)$$

avec $\langle dU, dW \rangle = \rho\,dt$, $k_1 > k_2$ ($X_t$ = facteur court, $Y_t$ = facteur long).

$$\boxed{\xi^T(t) = \xi^T(0)\exp\!\left(\omega\left[e^{-k_1\tau}X_t + \theta\,e^{-k_2\tau}Y_t\right] - \frac{\omega^2}{2}\left[e^{-2k_1\tau}\mathbb{E}[X_t^2] + \theta^2 e^{-2k_2\tau}\mathbb{E}[Y_t^2] + 2\theta\,e^{-(k_1+k_2)\tau}\mathbb{E}[X_tY_t]\right]\right)}$$

où $\tau = T - t$.

### 3.2 Récursions exactes

$$X_{t+\delta} = e^{-k_1\delta}X_t + x_\delta, \qquad \mathbb{E}[x_\delta^2] = \frac{1-e^{-2k_1\delta}}{2k_1}$$
$$Y_{t+\delta} = e^{-k_2\delta}Y_t + y_\delta, \qquad \mathbb{E}[y_\delta^2] = \frac{1-e^{-2k_2\delta}}{2k_2}$$
$$\mathbb{E}[X_{t+\delta}Y_{t+\delta}] = e^{-(k_1+k_2)\delta}\mathbb{E}[X_tY_t] + \rho\,\frac{1-e^{-(k_1+k_2)\delta}}{k_1+k_2}$$

In [ ]:
# ============================================================
#  MODÈLE À 2 FACTEURS — CLASSE COMPLÈTE
# ============================================================
class BergomiTwoFactor:
    """
    Modèle de Forward Variance à 2 facteurs (Bergomi 2005, section 2.2).

    dξ^T = ω ξ^T [e^{-k1(T-t)} dU_t + θ e^{-k2(T-t)} dW_t]
    corr(dU, dW) = ρ dt

    Paramètres
    ----------
    omega : float  — vol-de-vol globale
    k1    : float  — mean-reversion facteur court
    k2    : float  — mean-reversion facteur long
    theta : float  — poids relatif du facteur long
    rho   : float  — corrélation entre U et W
    """
    def __init__(self, omega, k1, k2, theta, rho):
        self.omega = omega
        self.k1    = k1
        self.k2    = k2
        self.theta = theta
        self.rho   = rho

    def xi(self, T: float, t: float, state: dict, xi0: float) -> float:
        """
        ξ^T(t) — équation (2.2).
        state = {'X': float, 'Y': float, 'EX2': float, 'EY2': float, 'EXY': float}
        """
        tau = T - t
        X, Y = state['X'], state['Y']
        EX2, EY2, EXY = state['EX2'], state['EY2'], state['EXY']
        w  = self.omega
        k1, k2, th = self.k1, self.k2, self.theta
        linear = w * (np.exp(-k1 * tau) * X + th * np.exp(-k2 * tau) * Y)
        quad   = 0.5 * w**2 * (
            np.exp(-2*k1*tau) * EX2
            + th**2 * np.exp(-2*k2*tau) * EY2
            + 2 * th * np.exp(-(k1+k2)*tau) * EXY
        )
        return xi0 * np.exp(linear - quad)

    def init_state(self) -> dict:
        return {'X': 0., 'Y': 0., 'EX2': 0., 'EY2': 0., 'EXY': 0.}

    def step_state(self, state: dict, delta: float,
                   rng: np.random.Generator) -> dict:
        """Pas de simulation exact pour les 5 variables d'état."""
        k1, k2, rho = self.k1, self.k2, self.rho
        # Incréments
        var_x   = (1 - np.exp(-2*k1*delta)) / (2*k1)
        var_y   = (1 - np.exp(-2*k2*delta)) / (2*k2)
        cov_xy  = rho * (1 - np.exp(-(k1+k2)*delta)) / (k1+k2)
        # Simulation corrélée (Cholesky)
        z1 = rng.standard_normal()
        z2 = rng.standard_normal()
        rho_eff = cov_xy / np.sqrt(var_x * var_y) if var_x > 0 and var_y > 0 else 0.
        rho_eff = np.clip(rho_eff, -1+1e-9, 1-1e-9)
        x_inc   = np.sqrt(var_x) * z1
        y_inc   = np.sqrt(var_y) * (rho_eff * z1 + np.sqrt(1 - rho_eff**2) * z2)
        # Mise à jour
        X_new  = np.exp(-k1*delta) * state['X'] + x_inc
        Y_new  = np.exp(-k2*delta) * state['Y'] + y_inc
        EX2    = np.exp(-2*k1*delta) * state['EX2'] + var_x
        EY2    = np.exp(-2*k2*delta) * state['EY2'] + var_y
        EXY    = np.exp(-(k1+k2)*delta) * state['EXY'] + cov_xy
        return {'X': X_new, 'Y': Y_new, 'EX2': EX2, 'EY2': EY2, 'EXY': EXY}

    def vol_of_vs_vol(self, tau: float, dt: float = 1/12) -> float:
        """
        Vol annualisée de ln(σ_VS(τ)) sur horizon dt.
        Pour le modèle 2F :
        Var[Δ ln ξ^T] ≈ ω² * [e^{-2k1τ} V_x + θ² e^{-2k2τ} V_y + 2θρ e^{-(k1+k2)τ} C_xy]
        """
        k1, k2, th = self.k1, self.k2, self.theta
        Vx  = (1 - np.exp(-2*k1*dt)) / (2*k1)
        Vy  = (1 - np.exp(-2*k2*dt)) / (2*k2)
        Cxy = self.rho * (1 - np.exp(-(k1+k2)*dt)) / (k1+k2)
        var_ln_xi = self.omega**2 * (
            np.exp(-2*k1*tau) * Vx
            + th**2 * np.exp(-2*k2*tau) * Vy
            + 2 * th * np.exp(-(k1+k2)*tau) * Cxy
        )
        return np.sqrt(var_ln_xi / dt)  # annualisé


m2f = BergomiTwoFactor(OMEGA_REF, K1_REF, K2_REF, THETA_REF, RHO_REF)
print('Modèle 2 facteurs instancié.')
print(f'Paramètres : ω={OMEGA_REF}, k₁={K1_REF}, k₂={K2_REF}, θ={THETA_REF}, ρ={RHO_REF}')

---
## Section 4 — Modèle à N facteurs

### 4.1 Spécification (équation 2.4)

On modélise les FV discrètes $\xi_i(t)$ pour les intervalles $[T_i, T_i+\Delta]$ :

$$\xi_i(t) = \xi_i(0)\,e^{\omega Z^i_t - \omega^2 t/2}$$

avec une **structure de corrélation** entre les $Z^i$ donnée par :

$$\boxed{\rho(Z^i, Z^j) = \theta\rho_0 + (1-\theta)\beta^{|j-i|}}$$

où $\theta, \rho_0, \beta \in [0,1]$.

**Différence clé avec le modèle 2F :** Le nombre de facteurs est proportionnel à $T$ → temps de calcul en $O(T^2)$.

In [ ]:
# ============================================================
#  MODÈLE N FACTEURS
# ============================================================
class BergomiNFactor:
    """
    Modèle de Forward Variance N facteurs (Bergomi 2005, section 2.4).

    ξ_i(t) = ξ_i(0) * exp(ω Z^i_t - ω² t/2)
    ρ(Z^i, Z^j) = θ ρ₀ + (1-θ) β^|j-i|
    """
    def __init__(self, omega, theta, rho0, beta, delta=1/12):
        self.omega = omega
        self.theta = theta
        self.rho0  = rho0
        self.beta  = beta
        self.delta = delta

    def corr_matrix(self, N: int) -> np.ndarray:
        """Matrice de corrélation N×N des processus Z^i."""
        i_idx = np.arange(N)
        dist  = np.abs(i_idx[:, None] - i_idx[None, :])
        return self.theta * self.rho0 + (1 - self.theta) * self.beta**dist

    def chol_factor(self, N: int) -> np.ndarray:
        """Facteur de Cholesky de la matrice de corrélation."""
        C = self.corr_matrix(N)
        # Regularisation numérique
        C += 1e-9 * np.eye(N)
        return np.linalg.cholesky(C)

    def vol_of_vs_vol(self, tau_months: np.ndarray,
                      dt_months: float = 1.0) -> np.ndarray:
        """
        Vol de vol pour maturité τ (en mois) mesurée sur dt mois.
        Dans le modèle N-facteurs (processus lognormal stationnaire) :
        Vol²(τ, dt) = ω² * (1 - e^{-2ω²dt}) * Corr(Z^τ_t, Z^τ_t)
        Simplified : ω² * dt (diagonal = 1)
        """
        # vol annualisée = ω * sqrt(dt/dt) = ω (constante par construction NF)
        # Mais la corrélation inter-facteurs atténue la vol pour les longs τ
        # Vol(τ, dt) sur 1 mois ≈ ω (constante, car chaque facteur a vol=ω)
        return np.ones_like(tau_months, dtype=float) * self.omega


# Paramètres N-facteurs de Bergomi
SIGMA_NF  = 2.40   # 240%
THETA_NF  = 0.40
RHO0_NF   = 0.05
BETA_NF   = 0.10

mNf = BergomiNFactor(SIGMA_NF, THETA_NF, RHO0_NF, BETA_NF)

# Affichage de la matrice de corrélation (exemple 12 mois)
corr_12 = mNf.corr_matrix(12)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_12, vmin=0, vmax=1, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_title('Matrice de corrélation N-facteurs — ξ_i vs ξ_j (12 mois)\n'
             f'θ={THETA_NF}, ρ₀={RHO0_NF}, β={BETA_NF}')
ax.set_xlabel('Mois j')
ax.set_ylabel('Mois i')
for i in range(12):
    for j in range(12):
        ax.text(j, i, f'{corr_12[i,j]:.2f}', ha='center', va='center',
                fontsize=7, color='white' if corr_12[i,j] > 0.5 else 'black')
plt.tight_layout()
plt.show()

---
## Section 5 — Spécification du processus spot : CEV discret

### 5.1 Dynamique sur l'intervalle $[T_i, T_i+\Delta]$ (équation 3.1)

Sur chaque intervalle $[T_i, T_i+\Delta]$, on utilise une **local volatility de type CEV** :

$$\boxed{dS = (r-q)S\,dt + \sigma_0(\hat{\sigma}_{VS}) \left(\frac{S}{S_{T_i}}\right)^{1-\beta(\hat{\sigma}_{VS})} S\,dZ_t}$$

Les fonctions $\sigma_0(\hat{\sigma}_{VS})$ et $\beta(\hat{\sigma}_{VS})$ sont calibrées de sorte que :
1. La **VS volatilité** de maturité $T_i+\Delta$ soit égale à $\hat{\sigma}_{VS} = \sqrt{\xi_i(T_i)}$
2. Le **skew ATMF** pour maturité $\Delta$ soit une valeur cible (ici constant à 5%)

### 5.2 Calibration de σ₀ et β

Pour la CEV, les moments de $\ln(S_{T_i+\Delta}/S_{T_i})$ sont donnés par :
$$\hat{\sigma}_{ATMF}^2 \approx \sigma_0^2 S^{2(1-\beta)}, \qquad \text{Skew}_{ATMF} \approx -(1-\beta)\sigma_0$$

Le skew 95%-105% se lit directement :
$$\hat{\sigma}_{95\%} - \hat{\sigma}_{105\%} \approx -\frac{1}{10}\left.\frac{d\hat{\sigma}_K}{d\ln K}\right|_F$$

In [ ]:
# ============================================================
#  CALIBRATION σ₀(σ_VS) ET β(σ_VS) POUR UN SKEW CIBLE
# ============================================================
def cev_moments(sigma0, beta_cev, S0, T, n_mc=50_000, seed=42):
    """
    Simule le processus CEV par Euler-Maruyama et retourne
    la VS vol et le skew 95%-105% implicites.
    """
    rng    = np.random.default_rng(seed)
    n_steps = max(20, int(T * 252))
    dt      = T / n_steps
    S       = np.full(n_mc, float(S0))
    var_acc = np.zeros(n_mc)

    for _ in range(n_steps):
        Z     = rng.standard_normal(n_mc)
        vol_t = sigma0 * (S / S0)**(1 - beta_cev)
        vol_t = np.clip(vol_t, 1e-4, 10.)
        var_acc += vol_t**2 * dt
        S = S * np.exp(-0.5 * vol_t**2 * dt + vol_t * np.sqrt(dt) * Z)
        S = np.maximum(S, 1e-6)

    vs_vol  = np.sqrt(np.mean(var_acc) / T)

    # Prix des options pour calculer le skew 95%-105%
    K95, K105 = S0 * 0.95, S0 * 1.05
    px95  = np.maximum(K95  - S, 0).mean()
    px105 = np.maximum(S - K105, 0).mean()
    iv95  = implied_vol(S0, K95,  T, max(px95,  1e-10), option='put')
    iv105 = implied_vol(S0, K105, T, max(px105, 1e-10), option='call')
    skew  = (iv95 - iv105) if (iv95 and iv105) else np.nan

    return vs_vol, skew


def calibrate_sigma0_beta(sigma_vs_target, skew_target,
                           S0=100., T=1/12, n_mc=30_000):
    """
    Trouve σ₀ et β tels que :
    - VS vol ≈ sigma_vs_target
    - Skew 95%-105% ≈ skew_target
    """
    def objective(params):
        sigma0, beta_cev = params
        if sigma0 <= 0 or not (0 <= beta_cev <= 1):
            return 1e6
        vs, sk = cev_moments(sigma0, beta_cev, S0, T, n_mc=n_mc)
        if not np.isfinite(vs) or not np.isfinite(sk):
            return 1e6
        err_vs   = (vs  - sigma_vs_target)**2 * 100
        err_skew = (sk  - skew_target)**2 * 10000
        return err_vs + err_skew

    x0  = [sigma_vs_target * 1.05, 0.5]
    res = minimize(objective, x0, method='Nelder-Mead',
                   options={'xatol': 1e-4, 'fatol': 1e-6, 'maxiter': 500})
    return res.x[0], res.x[1]


# Construction de la fonction σ₀(σ_VS), β(σ_VS)
# Grille de niveaux de volatilité (comme Figure 4.3 du papier)
SKEW_TARGET = 0.05   # skew 95%-105% constant à 5%
DELTA       = 1/12   # maturité 1 mois

sigma_vs_grid = np.array([0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60])

calib_path = cache_dir / 'cev_calib_grid.pkl'
if calib_path.exists():
    with open(calib_path, 'rb') as f:
        calib_cev = pickle.load(f)
    print('Cache calibration CEV chargé.')
else:
    print('Calibration CEV en cours...')
    sigma0_list, beta_list = [], []
    for svs in sigma_vs_grid:
        s0, b = calibrate_sigma0_beta(svs, SKEW_TARGET, n_mc=20_000)
        sigma0_list.append(s0)
        beta_list.append(b)
        print(f'  σ_VS={svs*100:.0f}% → σ₀={s0*100:.2f}%, β={b:.4f}')
    calib_cev = {'sigma_vs': sigma_vs_grid, 'sigma0': sigma0_list, 'beta': beta_list}
    with open(calib_path, 'wb') as f:
        pickle.dump(calib_cev, f)

sigma0_arr = np.array(calib_cev['sigma0'])
beta_arr   = np.array(calib_cev['beta'])

In [ ]:
# ============================================================
#  FIGURE 4.3 — σ₀ et β en fonction de σ_VS
# ============================================================
fig, ax1 = plt.subplots(figsize=(10, 5))

color1, color2 = 'steelblue', 'firebrick'
ax1.plot(sigma_vs_grid * 100, beta_arr, 'o-', color=color1, lw=2, ms=7, label='β (axe gauche)')
ax1.set_xlabel('σ_VS (%)')
ax1.set_ylabel('β', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(sigma_vs_grid * 100, sigma0_arr * 100, 's--', color=color2, lw=2, ms=7, label='σ₀ (axe droit)')
ax2.set_ylabel('σ₀ (%)', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title('Figure 4.3 — Calibration CEV : σ₀ et β en fonction de σ_VS\n'
              f'(skew 95%–105% constant = {SKEW_TARGET*100:.0f}%)')

plt.tight_layout()
plt.show()

# Tableau récapitulatif
pd.DataFrame({
    'σ_VS (%)': (sigma_vs_grid * 100).round(0).astype(int),
    'σ₀ (%)':   (sigma0_arr * 100).round(3),
    'β':        beta_arr.round(4),
    'Skew cible (%)': SKEW_TARGET * 100,
}).set_index('σ_VS (%)')

---
## Section 6 — Algorithme de simulation Monte Carlo complet

### 6.1 Équation de pricing (modèle 2 facteurs)

$$\frac{\partial P}{\partial t} + (r-q)S\frac{\partial P}{\partial S} - k_1 X\frac{\partial P}{\partial X} - k_2 Y\frac{\partial P}{\partial Y} + \frac{\sigma^2 S^2}{2}\frac{\partial^2 P}{\partial S^2}$$
$$+ \frac{1}{2}\left(\frac{\partial^2 P}{\partial X^2} + \frac{\partial^2 P}{\partial Y^2} + 2\rho\frac{\partial^2 P}{\partial X\partial Y}\right) + \sigma(\cdots, S)\,S\left(\rho_{SX}\frac{\partial^2 P}{\partial S\partial X} + \rho_{SY}\frac{\partial^2 P}{\partial S\partial Y}\right) = rP$$

### 6.2 Algorithme Monte Carlo — Structure temporelle discrète

Pour $T = N\Delta$ :

1. À $t = T_i$ : lire $\xi_i(T_i)$ → calculer $\hat{\sigma}_{VS} = \sqrt{\xi_i(T_i)}$
2. Lire $\sigma_0(\hat{\sigma}_{VS})$, $\beta(\hat{\sigma}_{VS})$ depuis la grille calibrée
3. Simuler $S_{T_{i+1}} / S_{T_i}$ par CEV sur $[T_i, T_i+\Delta]$, **corrélé** avec les innovations de $(X, Y)$
4. Mettre à jour $(X, Y, \mathbb{E}[X^2], \mathbb{E}[Y^2], \mathbb{E}[XY])$ → nouvelle courbe $\xi^T_{T_{i+1}}$

In [ ]:
# ============================================================
#  SIMULATEUR MONTE CARLO — MODÈLE BERGOMI 2 FACTEURS COMPLET
# ============================================================
from scipy.interpolate import interp1d

class BergomiMC:
    """
    Simulateur Monte Carlo complet du modèle Bergomi 2 facteurs.

    Paramètres du modèle :
    - Forward variance : ω, k1, k2, θ, ρ (entre U et W)
    - Spot/vol correlations : ρ_SX, ρ_SY
    - CEV spot : grille σ₀(σ_VS), β(σ_VS)
    - Tenor : Δ (par défaut 1 mois)
    """
    def __init__(self,
                 omega, k1, k2, theta, rho_uv,
                 rho_SX, rho_SY,
                 sigma_vs_grid, sigma0_grid, beta_grid,
                 xi0_flat: float = 0.04,   # variance (σ²_VS=20% → V=0.04)
                 delta: float = 1/12,
                 r: float = 0., q: float = 0.):
        self.omega   = omega
        self.k1      = k1
        self.k2      = k2
        self.theta   = theta
        self.rho_uv  = rho_uv
        self.rho_SX  = rho_SX
        self.rho_SY  = rho_SY
        self.delta   = delta
        self.r       = r
        self.q       = q
        self.xi0     = xi0_flat   # VS variance initiale (flat curve)
        # Interpolateurs pour la calibration CEV
        self._sig0_interp = interp1d(sigma_vs_grid, sigma0_grid,
                                     kind='linear', fill_value='extrapolate')
        self._beta_interp  = interp1d(sigma_vs_grid, beta_grid,
                                     kind='linear', fill_value='extrapolate')

    def _sigma0_beta(self, sigma_vs):
        sigma_vs = np.clip(sigma_vs, 0.05, 0.80)
        return float(self._sig0_interp(sigma_vs)), float(self._beta_interp(sigma_vs))

    def _xi_from_state(self, tau, state):
        """ξ^{t+τ}(t) — variante 2F de l'équation (2.2)."""
        w, k1, k2, th = self.omega, self.k1, self.k2, self.theta
        X, Y = state['X'], state['Y']
        EX2, EY2, EXY = state['EX2'], state['EY2'], state['EXY']
        linear = w * (np.exp(-k1 * tau) * X + th * np.exp(-k2 * tau) * Y)
        quad   = 0.5 * w**2 * (
            np.exp(-2*k1*tau) * EX2
            + th**2 * np.exp(-2*k2*tau) * EY2
            + 2 * th * np.exp(-(k1+k2)*tau) * EXY
        )
        return self.xi0 * np.exp(linear - quad)

    def _step_state(self, state, delta, rng):
        """Mise à jour exacte des 5 variables d'état + génère (xδ, yδ)."""
        k1, k2 = self.k1, self.k2
        rho     = self.rho_uv
        Vx  = (1 - np.exp(-2*k1*delta)) / (2*k1)
        Vy  = (1 - np.exp(-2*k2*delta)) / (2*k2)
        Cxy = rho * (1 - np.exp(-(k1+k2)*delta)) / (k1+k2)
        # Corrélation effective entre x_δ et y_δ
        rho_eff = np.clip(Cxy / (np.sqrt(Vx*Vy) + 1e-12), -1+1e-9, 1-1e-9)
        z1 = rng.standard_normal()
        z2 = rng.standard_normal()
        x_d = np.sqrt(Vx) * z1
        y_d = np.sqrt(Vy) * (rho_eff * z1 + np.sqrt(1 - rho_eff**2) * z2)
        X_n = np.exp(-k1*delta)*state['X'] + x_d
        Y_n = np.exp(-k2*delta)*state['Y'] + y_d
        EX2 = np.exp(-2*k1*delta)*state['EX2'] + Vx
        EY2 = np.exp(-2*k2*delta)*state['EY2'] + Vy
        EXY = np.exp(-(k1+k2)*delta)*state['EXY'] + Cxy
        return ({'X': X_n,'Y': Y_n,'EX2': EX2,'EY2': EY2,'EXY': EXY},
                x_d, y_d, np.sqrt(Vx), np.sqrt(Vy))

    def simulate(self, N_tenors: int, n_paths: int,
                 n_substeps: int = 10, seed: int = 42):
        """
        Simule N_tenors pas de longueur delta.
        Retourne un dict de trajectoires.
        """
        rng   = np.random.default_rng(seed)
        S0    = 100.
        delta = self.delta

        # Stockage
        S_paths   = np.zeros((n_paths, N_tenors + 1))
        xi0_paths = np.zeros((n_paths, N_tenors))   # ξ_i(T_i)
        S_paths[:, 0] = S0

        # État initial
        states = [{'X': 0., 'Y': 0., 'EX2': 0., 'EY2': 0., 'EXY': 0.}
                  for _ in range(n_paths)]

        # Corrélations spot/facteurs
        rSX, rSY = self.rho_SX, self.rho_SY
        # Construction de la matrice de Cholesky 3×3 pour (Z, U, W)
        # Z = spot brownien, U = facteur court, W = facteur long
        # corr(Z,U) = rSX, corr(Z,W) = rSY, corr(U,W) = rho_uv
        corr_3 = np.array([
            [1.,     rSX,            rSY],
            [rSX,    1.,             self.rho_uv],
            [rSY,    self.rho_uv,    1.]
        ])
        # Regularisation
        eigvals = np.linalg.eigvalsh(corr_3)
        if eigvals.min() < 0:
            corr_3 += (-eigvals.min() + 1e-8) * np.eye(3)
        L = np.linalg.cholesky(corr_3)

        dt_sub = delta / n_substeps

        for i in range(N_tenors):
            # Générer les innovations du facteur FV pour cette période
            dx_tot = np.zeros(n_paths)
            dy_tot = np.zeros(n_paths)
            xi_i   = np.zeros(n_paths)

            for p in range(n_paths):
                # ξ_i(T_i) = forward variance pour ce tenor
                xi_i[p] = self._xi_from_state(delta/2, states[p])  # approx milieu

            # CEV parameters for this tenor
            sigma_vs_i = np.sqrt(np.maximum(xi_i, 1e-6))

            # Simulation spot sur [T_i, T_i+Δ] en n_substeps
            S_curr = S_paths[:, i].copy()
            S_Ti   = S_curr.copy()   # niveau de référence CEV

            for _ in range(n_substeps):
                Z3 = rng.standard_normal((n_paths, 3)) @ L.T
                Z_spot = Z3[:, 0]   # bruit spot
                Z_X    = Z3[:, 1]   # bruit facteur court
                Z_Y    = Z3[:, 2]   # bruit facteur long

                # Vol locale CEV
                sig0 = np.array([float(self._sig0_interp(np.clip(sv, 0.05, 0.80)))
                                  for sv in sigma_vs_i])
                bet  = np.array([float(self._beta_interp(np.clip(sv, 0.05, 0.80)))
                                  for sv in sigma_vs_i])
                ratio   = np.maximum(S_curr / S_Ti, 1e-6)
                loc_vol = sig0 * ratio**(1 - bet)
                loc_vol = np.clip(loc_vol, 1e-4, 5.)

                S_curr = S_curr * np.exp(
                    (self.r - self.q - 0.5*loc_vol**2) * dt_sub
                    + loc_vol * np.sqrt(dt_sub) * Z_spot
                )
                dx_tot += Z_X * np.sqrt(dt_sub)
                dy_tot += Z_Y * np.sqrt(dt_sub)

            S_paths[:, i+1] = S_curr
            xi0_paths[:, i] = xi_i

            # Mise à jour des états (one-step exact)
            for p in range(n_paths):
                new_state, _, _, _, _ = self._step_state(states[p], delta, rng)
                states[p] = new_state

        monthly_returns = S_paths[:, 1:] / S_paths[:, :-1] - 1
        log_returns     = np.log(S_paths[:, 1:] / S_paths[:, :-1])

        return {
            'S': S_paths,
            'monthly_returns': monthly_returns,
            'log_returns': log_returns,
            'xi0': xi0_paths,
        }


# Paramètres par défaut (reproduire les exemples du papier)
RHO_SX_REF = -0.70
RHO_SY_REF = -0.357   # χ = -50%

mc_engine = BergomiMC(
    omega    = OMEGA_REF,
    k1       = K1_REF,
    k2       = K2_REF,
    theta    = THETA_REF,
    rho_uv   = RHO_REF,
    rho_SX   = RHO_SX_REF,
    rho_SY   = RHO_SY_REF,
    sigma_vs_grid = sigma_vs_grid,
    sigma0_grid   = sigma0_arr,
    beta_grid     = beta_arr,
    xi0_flat = 0.04,   # 20% annualisée
    delta    = 1/12,
)

print('Moteur MC Bergomi 2F instancié.')
print('Test rapide (1000 paths, 6 tenors)...')
test_sim = mc_engine.simulate(N_tenors=6, n_paths=1_000, n_substeps=5, seed=0)
print(f'S final moyen : {test_sim["S"][:,-1].mean():.2f} (attendu ~100)')
print(f'Vol réalisée  : {test_sim["log_returns"].std(axis=0).mean() * np.sqrt(12) * 100:.2f}%')

---
## Section 7 — Structure par terme de la vol-de-vol

### 7.1 Comparaison modèle 2F vs N facteurs

Bergomi compare les deux modèles sur **deux horizons de mesure** : 1 mois et 1 an.

**Différence clé :**
- **Modèle 2F** : la vol-de-vol *décroît* avec l'horizon (mean-reversion des facteurs OU)
- **Modèle NF** : la vol-de-vol *croît* avec l'horizon (lognormalité des forward variances)

On calcule :
$$\text{Vol-de-vol}(\tau, \Delta t) = \frac{1}{\sqrt{\Delta t}}\,\text{Std}\!\left[\ln\frac{\hat{\sigma}_{VS}(\tau; t+\Delta t)}{\hat{\sigma}_{VS}(\tau; t)}\right]$$

In [ ]:
# ============================================================
#  STRUCTURE PAR TERME DE LA VOL-DE-VOL (Figures 4.1 & 4.2)
# ============================================================
taus_months = np.array([1, 3, 6, 12, 24, 36, 48, 60])
taus_years  = taus_months / 12.

# --- MODÈLE 2 FACTEURS ---
def volvol_2f_dt(omega, k1, k2, theta, rho, tau_y, dt_y):
    """
    Vol annualisée de ln(σ_VS(τ)) sur horizon dt — modèle 2 facteurs.
    Var[Δ ln ξ^T] = ω² * [
       e^{-2k1τ} * (1-e^{-2k1dt})/(2k1)
     + θ² e^{-2k2τ} * (1-e^{-2k2dt})/(2k2)
     + 2θρ e^{-(k1+k2)τ} * (1-e^{-(k1+k2)dt})/(k1+k2)
    ]
    """
    Vx  = (1 - np.exp(-2*k1*dt_y)) / (2*k1)
    Vy  = (1 - np.exp(-2*k2*dt_y)) / (2*k2)
    Cxy = rho * (1 - np.exp(-(k1+k2)*dt_y)) / (k1+k2)
    var = omega**2 * (
        np.exp(-2*k1*tau_y) * Vx
        + theta**2 * np.exp(-2*k2*tau_y) * Vy
        + 2 * theta * np.exp(-(k1+k2)*tau_y) * Cxy
    )
    return np.sqrt(var / dt_y)   # annualisé


# --- MODÈLE N FACTEURS ---
# Dans le modèle NF, chaque ξ_i est lognormal avec vol ω*sqrt(dt),
# et les corrélations croisées réduisent la vol de la VS pour les longs τ.
# Approximation analytique :
def volvol_nf_dt(omega_nf, theta, rho0, beta, tau_months, dt_months):
    """
    Vol-de-vol approximée pour le modèle N-facteurs.
    Pour une VS de maturité τ = n mois, on moyenne les ξ_i sur i=0..n-1.
    La variance de ln(VS_var) est :
    Var ≈ ω² * dt * (1/n²) * Σ_{i,j} ρ(Z^i, Z^j)
    """
    n = int(tau_months)
    if n < 1: n = 1
    # Somme des corrélations
    i_arr, j_arr = np.meshgrid(np.arange(n), np.arange(n))
    corr_sum = np.sum(theta * rho0 + (1 - theta) * beta**np.abs(i_arr - j_arr))
    var_ln_vs = omega_nf**2 * (dt_months / 12.) * corr_sum / n**2
    return np.sqrt(var_ln_vs / (dt_months / 12.))


fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_idx, (dt_months, title_dt) in enumerate([(1, 'Δt = 1 mois'), (12, 'Δt = 1 an')]):
    dt_y = dt_months / 12.

    vv_2f = [volvol_2f_dt(OMEGA_REF, K1_REF, K2_REF, THETA_REF, RHO_REF,
                           t, dt_y) for t in taus_years]
    vv_nf = [volvol_nf_dt(SIGMA_NF, THETA_NF, RHO0_NF, BETA_NF,
                           t, dt_months) for t in taus_months]

    axes[ax_idx].plot(taus_months, np.array(vv_2f) * 100, 'o-',
                      color='steelblue', lw=2, ms=7, label='Modèle 2 facteurs')
    axes[ax_idx].plot(taus_months, np.array(vv_nf) * 100, 's--',
                      color='firebrick', lw=2, ms=7, label='Modèle N facteurs')
    axes[ax_idx].axhline(OMEGA_REF * 100, ls=':', color='gray', lw=1, label=f'ω = {OMEGA_REF*100:.0f}%')
    axes[ax_idx].set_xlabel('Maturité de la VS vol (mois)')
    axes[ax_idx].set_ylabel('Vol de VS vol (%, annualisée)')
    axes[ax_idx].set_title(f'Figures 4.1 & 4.2 — Vol-de-vol — {title_dt}')
    axes[ax_idx].legend()
    axes[ax_idx].set_ylim(0, 200)

plt.suptitle('Structure par terme de la Vol-de-Vol\n(Bergomi 2005, paramètres de référence)',
             fontweight='bold')
plt.tight_layout()
plt.show()

print('Observations clés :')
print('  Δt=1M : les deux modèles sont similaires (calibration commune)')
print('  Δt=1Y : le modèle NF donne des vol-de-vol plus élevées (lognormalité)')
print('         → impact significatif sur les options de variance longue maturité')

In [ ]:
# ============================================================
#  CALIBRATION ω (vol-de-vol) : cible 120% à 1 mois
# ============================================================
# Bergomi impose : Vol(VS 1M, Δt=1M) = 120%
# → ω * e^{-k1*(1/12)} * sqrt((1-e^{-2k1/12})/(2k1)) / sqrt(1/12) = 1.20

def omega_for_target_volvol(target=1.20, k1=K1_REF, dt=1/12, tau=1/12):
    """Calcule ω pour atteindre une vol-de-vol cible à court terme."""
    Vx = (1 - np.exp(-2*k1*dt)) / (2*k1)
    factor = np.exp(-k1 * tau) * np.sqrt(Vx / dt)
    return target / factor

omega_calibrated = omega_for_target_volvol(1.20)
print(f'ω calibré pour vol-de-vol 1M = 120% : ω = {omega_calibrated:.4f}')
print(f'Bergomi utilise ω = {OMEGA_REF} (différence : {abs(omega_calibrated - OMEGA_REF):.4f})')

# Tableau de sensibilité
print('\nSensibilité de la Vol-de-Vol 1M à ω :')
omega_grid = np.array([1.5, 2.0, 2.5, 2.827, 3.0, 3.5])
sens_df = pd.DataFrame({
    'ω': omega_grid,
    'VV 1M (%)': [volvol_2f_dt(w, K1_REF, K2_REF, THETA_REF, RHO_REF, 1/12, 1/12)*100 for w in omega_grid],
    'VV 1Y (%)': [volvol_2f_dt(w, K1_REF, K2_REF, THETA_REF, RHO_REF, 1., 1/12)*100 for w in omega_grid],
    'VV 5Y (%)': [volvol_2f_dt(w, K1_REF, K2_REF, THETA_REF, RHO_REF, 5., 1/12)*100 for w in omega_grid],
}).round(1).set_index('ω')
print(sens_df)

---
## Section 8 — Structure par terme du skew

### 8.1 Formule analytique (équation 4.4)

À l'ordre 1 en $\omega$ et en $\text{Skew}_\Delta$, le skew ATMF pour maturité $T = N\Delta$ est :

$$\boxed{\text{Skew}_{N\Delta} = \frac{\text{Skew}_\Delta}{N} + \omega \cdot \sqrt{2}\left[\rho_{SX}\,\zeta(k_1\Delta, N) + \theta\rho_{SY}\,\zeta(k_2\Delta, N)\right]}$$

où la fonction $\zeta$ capture la contribution de la corrélation spot/vol :

$$\zeta(x, N) = \frac{1-e^{-x}}{x} \cdot \frac{\sum_{\tau=1}^{N-1}(N-\tau)e^{-(\tau-1)x}}{N^2}$$

### 8.2 Décomposition intrinsèque / corrélation spot-vol

- **Terme intrinsèque** $\text{Skew}_\Delta / N$ : décroît comme $1/T$ (indépendance des incréments)
- **Terme spot/vol** : non-monotone, nul à $T=0$, puis décroît comme $1/T$ pour $T \gg 1/k$

In [ ]:
# ============================================================
#  FONCTION ζ ET FORMULE DU SKEW (équation 4.3 & 4.4)
# ============================================================
def zeta(x: float, N: int) -> float:
    """Équation (4.3) du papier."""
    if x < 1e-8 or N <= 1:
        return 0.
    tau_arr = np.arange(1, N)
    num = np.sum((N - tau_arr) * np.exp(-(tau_arr - 1) * x))
    prefactor = (1 - np.exp(-x)) / x
    return prefactor * num / N**2


def skew_term_structure(Skew_delta, omega, k1, k2, theta, rho_SX, rho_SY,
                         N_max: int = 60, delta: float = 1/12):
    """
    Structure par terme du skew selon l'équation (4.4).
    Retourne (maturités en mois, skew total, skew intrinsèque, skew spot/vol)
    """
    N_arr     = np.arange(1, N_max + 1)
    mats      = N_arr * delta * 12   # en mois
    intrinsic = Skew_delta / N_arr

    # Le facteur √2 vient de la normalisation des corrélations
    spotvol_contrib = omega * np.sqrt(2) * np.array([
        rho_SX * zeta(k1 * delta, N) + theta * rho_SY * zeta(k2 * delta, N)
        for N in N_arr
    ])

    total = intrinsic + spotvol_contrib
    return mats, total, intrinsic, spotvol_contrib


# Paramètres
SKEW_DELTA = 0.05   # skew 1 mois cible en fraction (5%)

mats, sk_total, sk_intr, sk_sv = skew_term_structure(
    SKEW_DELTA, OMEGA_REF, K1_REF, K2_REF, THETA_REF,
    RHO_SX_REF, RHO_SY_REF
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Figure 4.4 — Skew total (analytique vs MC)
axes[0].plot(mats, sk_total * 100, 'steelblue', lw=2.5, label='Formule (4.4) — analytique')
axes[0].axhline(SKEW_DELTA * 100, ls=':', color='gray', label=f'Skew∆ = {SKEW_DELTA*100:.0f}%')
axes[0].set_xlabel('Maturité (mois)')
axes[0].set_ylabel('Skew 95%–105% (%)')
axes[0].set_title('Figure 4.4 — Structure par terme du skew')
axes[0].legend()

# Figure 4.5 — Décomposition
axes[1].stackplot(mats,
                  sk_intr * 100,
                  sk_sv * 100,
                  labels=['Intrinsèque (Skew∆/N)', 'Spot/Vol corrélation'],
                  colors=['steelblue', 'firebrick'],
                  alpha=0.7)
axes[1].plot(mats, sk_total * 100, 'k-', lw=2, label='Total')
axes[1].set_xlabel('Maturité (mois)')
axes[1].set_ylabel('Contribution au skew (%)')
axes[1].set_title('Figure 4.5 — Décomposition du skew')
axes[1].legend()

plt.suptitle('Figures 4.4 & 4.5 — Structure par terme du skew (Bergomi 2005)',
             fontweight='bold')
plt.tight_layout()
plt.show()

# Tableau des valeurs clés
key_mats = [1, 3, 6, 12, 24, 36, 48, 60]
res = []
for m in key_mats:
    idx = np.argmin(np.abs(mats - m))
    res.append({'Maturité (mois)': m,
                'Skew total (%)': round(sk_total[idx]*100, 3),
                'Intrinsèque (%)': round(sk_intr[idx]*100, 3),
                'Spot/Vol (%)': round(sk_sv[idx]*100, 3)})
pd.DataFrame(res).set_index('Maturité (mois)')

In [ ]:
# ============================================================
#  SENSIBILITÉ AU SIGNE ET AMPLITUDE DE ρ_SX, ρ_SY
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Sensibilité à ρ_SX
for rSX in [-0.90, -0.70, -0.50, -0.30, 0.0]:
    _, sk, _, _ = skew_term_structure(SKEW_DELTA, OMEGA_REF, K1_REF, K2_REF,
                                      THETA_REF, rSX, RHO_SY_REF)
    axes[0].plot(mats, sk * 100, lw=2, label=f'ρ_SX = {rSX:.0%}')
axes[0].axhline(0, color='k', lw=0.8, ls='--')
axes[0].set_xlabel('Maturité (mois)')
axes[0].set_ylabel('Skew 95%–105% (%)')
axes[0].set_title('Impact de ρ_SX (facteur court) sur le skew')
axes[0].legend(fontsize=9)

# Sensibilité à θ (poids du facteur long)
for th in [0.0, 0.15, 0.30, 0.50, 0.70]:
    _, sk, _, _ = skew_term_structure(SKEW_DELTA, OMEGA_REF, K1_REF, K2_REF,
                                      th, RHO_SX_REF, RHO_SY_REF)
    axes[1].plot(mats, sk * 100, lw=2, label=f'θ = {th:.0%}')
axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel('Maturité (mois)')
axes[1].set_ylabel('Skew 95%–105% (%)')
axes[1].set_title('Impact de θ (poids facteur long) sur le skew')
axes[1].legend(fontsize=9)

plt.suptitle('Sensibilité de la structure par terme du skew', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 9 — Pricing d'options exotiques

### 9.1 Définition des produits

**Reverse Cliquet** (globalement flooré, localement cappé) :
$$P_{RC} = \max\!\left(0,\; C + \sum_{i=1}^{N} r_i^-\right), \qquad r_i^- = \min(r_i, 0)$$
avec $C = 50\%$, $N=36$ mois, 3 ans.

**Napoleon** :
$$P_{Nap,k} = \max\!\left(0,\; C + \min_{j \in \text{année }k} r_j\right), \qquad C = 8\%$$
3 ans, coupon annuel basé sur le minimum des 12 rendements mensuels.

**Accumulator** (capped & floored) :
$$P_{Acc} = \max\!\left(0,\; \sum_{i=1}^{N} \max(\min(r_i, \text{cap}), \text{floor})\right), \qquad \text{floor}=-1\%,\;\text{cap}=+1\%$$

### 9.2 Quatre configurations tarifaires

| Configuration | Forward Skew | Vol-de-Vol |
|---|---|---|
| Black-Scholes | ✗ | ✗ |
| Forward Skew only | ✓ | ✗ |
| Vol-de-Vol only | ✗ | ✓ |
| Full (Bergomi) | ✓ | ✓ |

In [ ]:
# ============================================================
#  FONCTIONS DE PAYOFF
# ============================================================
def payoff_reverse_cliquet(monthly_returns, C=0.50):
    """
    Reverse Cliquet : max(0, C + Σ min(r_i, 0))
    monthly_returns : shape (n_paths, N_months)
    """
    r_neg = np.minimum(monthly_returns, 0.)
    return np.maximum(C + r_neg.sum(axis=1), 0.)


def payoff_napoleon(monthly_returns, C=0.08, months_per_year=12):
    """
    Napoleon : Σ_k max(0, C + min_j r_{k,j})
    Coupon annuel basé sur le minimum mensuel de chaque année.
    """
    n_paths, N = monthly_returns.shape
    n_years    = N // months_per_year
    total      = np.zeros(n_paths)
    for k in range(n_years):
        annual_rets = monthly_returns[:, k*months_per_year:(k+1)*months_per_year]
        worst       = annual_rets.min(axis=1)
        total      += np.maximum(C + worst, 0.)
    return total


def payoff_accumulator(monthly_returns, floor=-0.01, cap=0.01):
    """
    Accumulator : max(0, Σ clamp(r_i, floor, cap))
    """
    clamped = np.clip(monthly_returns, floor, cap)
    return np.maximum(clamped.sum(axis=1), 0.)


print('Fonctions de payoff définies.')

In [ ]:
# ============================================================
#  SIMULATION MC POUR LES 4 CONFIGURATIONS
# ============================================================
N_TENORS  = 36    # 3 ans × 12 mois
N_PATHS   = 20_000
N_SUBSTEP = 8

exotic_cache = cache_dir / 'bergomi2_exotic_prices.pkl'

if exotic_cache.exists():
    with open(exotic_cache, 'rb') as f:
        results_table = pickle.load(f)
    print('Cache exotic prices chargé.')
else:
    print('Simulation MC en cours (peut prendre quelques minutes)...')

    # Config 1 : Black-Scholes (flat vol, no stochastic vol)
    def bs_mc_paths(vol_atm, N_tenors, n_paths, seed=42):
        """Simulation BS avec vol constante (pas de vol-de-vol, pas de skew fwd)."""
        rng = np.random.default_rng(seed)
        dt  = 1/12
        Z   = rng.standard_normal((n_paths, N_tenors))
        log_r = (-0.5 * vol_atm**2 * dt) + vol_atm * np.sqrt(dt) * Z
        return np.exp(log_r) - 1   # monthly returns

    VOL_ATM = 0.20
    print('  BS...')
    r_bs = bs_mc_paths(VOL_ATM, N_TENORS, N_PATHS)

    bs_prices = {
        'Reverse Cliquet': payoff_reverse_cliquet(r_bs).mean(),
        'Napoleon':        payoff_napoleon(r_bs).mean(),
        'Accumulator':     payoff_accumulator(r_bs).mean(),
    }

    # Config 2 : Forward Skew uniquement (ω=0, skew≠0)
    print('  Forward Skew only (ω=0)...')
    mc_skew_only = BergomiMC(
        omega=0.001, k1=K1_REF, k2=K2_REF, theta=THETA_REF, rho_uv=RHO_REF,
        rho_SX=RHO_SX_REF, rho_SY=RHO_SY_REF,
        sigma_vs_grid=sigma_vs_grid, sigma0_grid=sigma0_arr, beta_grid=beta_arr,
        xi0_flat=VOL_ATM**2, delta=1/12)
    sim_skew = mc_skew_only.simulate(N_TENORS, N_PATHS, N_SUBSTEP, seed=1)
    skew_prices = {
        'Reverse Cliquet': payoff_reverse_cliquet(sim_skew['monthly_returns']).mean(),
        'Napoleon':        payoff_napoleon(sim_skew['monthly_returns']).mean(),
        'Accumulator':     payoff_accumulator(sim_skew['monthly_returns']).mean(),
    }

    # Config 3 : Vol-de-Vol uniquement (skew=0, ω≠0)
    print('  Vol-de-Vol only (skew=0)...')
    # Calibration sans skew : β=0, σ₀=σ_VS
    sigma0_flat = sigma_vs_grid.copy()
    beta_flat   = np.zeros_like(sigma_vs_grid)
    mc_vvol_only = BergomiMC(
        omega=OMEGA_REF, k1=K1_REF, k2=K2_REF, theta=THETA_REF, rho_uv=RHO_REF,
        rho_SX=RHO_SX_REF, rho_SY=RHO_SY_REF,
        sigma_vs_grid=sigma_vs_grid, sigma0_grid=sigma0_flat, beta_grid=beta_flat,
        xi0_flat=VOL_ATM**2, delta=1/12)
    sim_vvol = mc_vvol_only.simulate(N_TENORS, N_PATHS, N_SUBSTEP, seed=2)
    vvol_prices = {
        'Reverse Cliquet': payoff_reverse_cliquet(sim_vvol['monthly_returns']).mean(),
        'Napoleon':        payoff_napoleon(sim_vvol['monthly_returns']).mean(),
        'Accumulator':     payoff_accumulator(sim_vvol['monthly_returns']).mean(),
    }

    # Config 4 : Full Bergomi
    print('  Full Bergomi...')
    sim_full = mc_engine.simulate(N_TENORS, N_PATHS, N_SUBSTEP, seed=3)
    full_prices = {
        'Reverse Cliquet': payoff_reverse_cliquet(sim_full['monthly_returns']).mean(),
        'Napoleon':        payoff_napoleon(sim_full['monthly_returns']).mean(),
        'Accumulator':     payoff_accumulator(sim_full['monthly_returns']).mean(),
    }

    results_table = {
        'Black-Scholes': bs_prices,
        'Forward Skew uniquement': skew_prices,
        'Vol-de-Vol uniquement': vvol_prices,
        'Full (Bergomi)': full_prices,
        'sims': {'skew': sim_skew, 'vvol': sim_vvol, 'full': sim_full}
    }
    with open(exotic_cache, 'wb') as f:
        pickle.dump(results_table, f)
    print('Cache sauvegardé.')

print('\nPrices computed OK.')

In [ ]:
# ============================================================
#  TABLEAU 5.1 — RÉPLIQUE
# ============================================================
rows = []
for model_name in ['Black-Scholes', 'Forward Skew uniquement',
                   'Vol-de-Vol uniquement', 'Full (Bergomi)']:
    row = {'Modèle': model_name}
    for prod in ['Reverse Cliquet', 'Napoleon', 'Accumulator']:
        row[prod] = f"{results_table[model_name][prod]*100:.2f}%"
    rows.append(row)

price_df = pd.DataFrame(rows).set_index('Modèle')

print('='*65)
print('  TABLE 5.1 — Prix des options exotiques (Bergomi 2005)')
print('='*65)
print(price_df.to_string())
print()
print('Bergomi (papier) :')
bergomi_ref = pd.DataFrame({
    'Reverse Cliquet': ['0.25%', '0.56%', '2.92%', '3.81%'],
    'Napoleon':        ['2.10%', '2.13%', '4.71%', '4.45%'],
    'Accumulator':     ['1.90%', '4.32%', '1.90%', '5.06%'],
}, index=['Black-Scholes', 'Forward Skew', 'Vol-de-Vol', 'Full'])
print(bergomi_ref.to_string())

In [ ]:
# ============================================================
#  VISUALISATION — CONTRIBUTION DE CHAQUE EFFET
# ============================================================
products    = ['Reverse Cliquet', 'Napoleon', 'Accumulator']
configs     = ['Black-Scholes', 'Forward Skew uniquement',
               'Vol-de-Vol uniquement', 'Full (Bergomi)']
colors_cfg  = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, prod in zip(axes, products):
    vals = [results_table[c][prod] * 100 for c in configs]
    bars = ax.bar(range(len(configs)), vals, color=colors_cfg, edgecolor='white', lw=1.5)
    ax.set_xticks(range(len(configs)))
    ax.set_xticklabels(['BS', 'Skew', 'VoV', 'Full'], fontsize=10)
    ax.set_ylabel('Prix (% notionnel)')
    ax.set_title(prod)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val:.2f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Tableau 5.1 — Impact du skew forward et de la vol-de-vol sur les exotiques',
             fontweight='bold')
plt.tight_layout()
plt.show()

# Analyse des contributions
print('\nAnalyse des contributions (vs Black-Scholes) :')
for prod in products:
    bs   = results_table['Black-Scholes'][prod]
    skew = results_table['Forward Skew uniquement'][prod]
    vvol = results_table['Vol-de-Vol uniquement'][prod]
    full = results_table['Full (Bergomi)'][prod]
    print(f'\n  {prod}:')
    print(f'    +Skew   : +{(skew-bs)*100:.2f}%')
    print(f'    +VoV    : +{(vvol-bs)*100:.2f}%')
    print(f'    Full    : +{(full-bs)*100:.2f}% (interaction : {(full-skew-vvol+bs)*100:.2f}%)')

### 9.3 Interprétation économique

**Reverse Cliquet :** La vol-de-vol est l'effet dominant (+VoV >> +Skew). C'est une **Put sur la volatilité** : la convexité en vol crée de la valeur. Le skew augmente le prix car il valorise les options de faible strike (les payoffs négatifs conditionnés aux marchés baissiers).

**Napoleon :** La vol-de-vol domine également. Le skew forward a un impact ambigu selon la taille du coupon $C$ : les deux strikes du call spread peuvent être sous le forward.

**Accumulator :** Le skew forward est l'effet dominant. La vol-de-vol a peu d'impact seule (call spread ATM ≈ zéro vega en BS), mais crée de la convexité en présence du skew.

---
## Section 10 — Options sur variance réalisée

### 10.1 Définition

Un **call sur variance** de strike $\hat{\sigma}_K$ paie :
$$P_{VarCall} = \frac{1}{2\hat{\sigma}_K}\max\!\left(\sigma_h^2 - \hat{\sigma}_K^2,\; 0\right)$$

où $\sigma_h^2$ est la variance réalisée annualisée (mesurée sur les rendements quotidiens).

La **variance de la distribution de $\sigma_h^2$** a deux sources :
1. **Dynamique des VS variances** : contrôlée par $\omega, k_1, k_2, \theta$
2. **Observations discrètes** : contribution de la kurtosis des rendements (dépend de $\beta$)

### 10.2 Impact des hypothèses sur le skew court terme

Pour les options courtes ($T < \Delta$), seul le second effet compte. La kurtosis conditionnelle dépend de $\beta$ via la CEV.

In [ ]:
# ============================================================
#  PRICING D'OPTIONS SUR VARIANCE RÉALISÉE
# ============================================================
def price_variance_call(monthly_returns, sigma_K, obs_per_year=12):
    """
    Prix d'un call sur variance annualisée.
    Payoff = max(sigma²_réal - sigma²_K, 0) / (2 * sigma_K)
    """
    n_paths, N = monthly_returns.shape
    # Variance réalisée annualisée
    log_ret    = np.log(1 + monthly_returns)
    var_real   = log_ret.var(axis=1) * obs_per_year
    payoff     = np.maximum(var_real - sigma_K**2, 0.) / (2 * sigma_K)
    return payoff.mean(), var_real


def implied_vol_var_call(price, S_vs, sigma_K, T_y):
    """
    Vol implicite d'un call sur variance (Black avec F = S_vs = VS variance).
    Underlying = σ²_VS, on convertit en vol BS standard.
    """
    if price <= 0:
        return np.nan
    # Forward = S_vs (VS variance initiale)
    # On cherche σ_impl t.q. BS_call(S_vs, sigma_K, T, σ_impl) = price
    try:
        iv = brentq(
            lambda v: bs_price(S_vs, sigma_K, T_y, v, option='call') - price,
            1e-4, 10., xtol=1e-8
        )
        return iv
    except Exception:
        return np.nan


# Grille de maturités
mat_months_var = [1, 2, 3, 6, 12, 18, 24]
SIGMA_K_ATM    = 0.20   # strike = 20% vol = 4% variance
VAR_K          = SIGMA_K_ATM**2

var_cache = cache_dir / 'bergomi2_var_options.pkl'

if var_cache.exists():
    with open(var_cache, 'rb') as f:
        var_results = pickle.load(f)
    print('Cache options variance chargé.')
else:
    print('Pricing options sur variance...')
    var_results = {}

    for config, (use_skew, use_vvol) in [
        ('Avec skew fwd', (True, True)),
        ('Sans skew fwd', (False, True))
    ]:
        ivs_config = []
        s0_grid = sigma0_arr if use_skew else sigma_vs_grid
        b_grid  = beta_arr   if use_skew else np.zeros_like(sigma_vs_grid)

        for n_months in mat_months_var:
            engine = BergomiMC(
                omega=OMEGA_REF, k1=K1_REF, k2=K2_REF,
                theta=THETA_REF, rho_uv=RHO_REF,
                rho_SX=RHO_SX_REF, rho_SY=RHO_SY_REF,
                sigma_vs_grid=sigma_vs_grid,
                sigma0_grid=s0_grid,
                beta_grid=b_grid,
                xi0_flat=SIGMA_K_ATM**2, delta=1/12)
            sim  = engine.simulate(n_months, 15_000, n_substeps=10,
                                   seed=42)
            px, _ = price_variance_call(sim['monthly_returns'], SIGMA_K_ATM)
            T_y  = n_months / 12.
            # Vol implicite sur underlying = VS vol
            iv   = implied_vol_var_call(px, SIGMA_K_ATM, SIGMA_K_ATM, T_y)
            ivs_config.append({'T_months': n_months, 'price': px, 'impl_vol': iv})
            print(f'  {config} — {n_months}M : price={px*100:.3f}%, IV={iv*100 if iv else np.nan:.1f}%')
        var_results[config] = pd.DataFrame(ivs_config)

    with open(var_cache, 'wb') as f:
        pickle.dump(var_results, f)
    print('Cache sauvegardé.')

In [ ]:
# ============================================================
#  FIGURE 5.1 — VOL IMPLICITE DES CALLS SUR VARIANCE
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))

colors_var = ['steelblue', 'firebrick']
for (config, df_var), color in zip(var_results.items(), colors_var):
    days = df_var['T_months'].values * 21  # jours trading
    ivs  = df_var['impl_vol'].values * 100
    ax.plot(days, np.where(np.isfinite(ivs), ivs, np.nan),
            'o-', color=color, lw=2, ms=8, label=config)

ax.set_xlabel('Maturité de l\'option (jours trading)')
ax.set_ylabel('Vol implicite (%, underlying = VS var)')
ax.set_title('Figure 5.1 — Vol implicite des calls sur variance réalisée\n'
             f'(Strike = {SIGMA_K_ATM*100:.0f}%, VS vol plate initiale = {SIGMA_K_ATM*100:.0f}%)')
ax.legend()
plt.tight_layout()
plt.show()

print('\nObservation clé (Bergomi section 5.6) :')
print('  Le skew forward augmente significativement la vol des options sur variance,')
print('  surtout pour les maturités courtes (< 1 mois), où seul β compte.')
print('  Pour les longues maturités, la dynamique des ξ_i prend le dessus.')

In [ ]:
# ============================================================
#  DISTRIBUTION DE LA VARIANCE RÉALISÉE — IMPACT DE β
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Simuler 1 mois de rendements avec et sans skew
sim_full_data = results_table['sims']['full']
sim_vvol_data = results_table['sims']['vvol']

log_r_full  = np.log(1 + sim_full_data['monthly_returns'])
log_r_vvol  = np.log(1 + sim_vvol_data['monthly_returns'])

# Variance réalisée annualisée par chemin
var_real_full = log_r_full.var(axis=1) * 12
var_real_vvol = log_r_vvol.var(axis=1) * 12

axes[0].hist(np.sqrt(var_real_full) * 100, bins=80, density=True,
             color='steelblue', alpha=0.7, label='Full (avec skew fwd)')
axes[0].hist(np.sqrt(var_real_vvol) * 100, bins=80, density=True,
             color='firebrick', alpha=0.5, label='Vol-de-vol seule (sans skew)')
axes[0].set_xlabel('Volatilité réalisée 3Y (%)')
axes[0].set_ylabel('Densité')
axes[0].set_title('Distribution de la vol réalisée (3 ans)')
axes[0].legend()

# Distribution des rendements mensuels
from scipy.stats import skew as scipy_skew, kurtosis as scipy_kurt
r_flat = log_r_full.flatten()
r_vvol = log_r_vvol.flatten()
x_grid = np.linspace(-0.15, 0.15, 300)

axes[1].hist(r_flat, bins=120, density=True, color='steelblue', alpha=0.6,
             label=f'Full (sk={scipy_skew(r_flat):.3f}, ku={scipy_kurt(r_flat):.2f})')
axes[1].hist(r_vvol, bins=120, density=True, color='firebrick', alpha=0.4,
             label=f'No skew (sk={scipy_skew(r_vvol):.3f}, ku={scipy_kurt(r_vvol):.2f})')
axes[1].plot(x_grid, norm.pdf(x_grid, 0, r_flat.std()), 'k--', lw=2, label='Normale')
axes[1].set_xlabel('Rendement mensuel')
axes[1].set_ylabel('Densité')
axes[1].set_title('Distribution des rendements mensuels')
axes[1].legend(fontsize=9)

plt.suptitle('Impact du skew forward sur la distribution des rendements', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 11 — Calibration sur données SX5E réelles

### 11.1 Extraction de la courbe ξ sur 5 ans

On extrait pour chaque date la courbe $T \mapsto \xi^T_t$ et on calibre les paramètres du modèle à 2 facteurs.

In [ ]:
# ============================================================
#  EXTRACTION DE ξ^T_t POUR TOUTES LES DATES
# ============================================================
xi_curves_cache = cache_dir / 'xi_curves_all_dates.pkl'

if xi_curves_cache.exists():
    with open(xi_curves_cache, 'rb') as f:
        xi_all = pickle.load(f)
    print(f'Cache ξ-curves chargé : {len(xi_all)} dates.')
else:
    print('Extraction des courbes ξ^T_t...')
    xi_all = {}
    dates  = surface['date'].unique()
    for i, d in enumerate(dates):
        vs  = extract_vs_curve(surface, d)
        if len(vs) >= 3:
            xi  = extract_xi_curve(vs)
            xi_all[d] = xi
        if (i+1) % 100 == 0:
            print(f'  {i+1}/{len(dates)}')
    with open(xi_curves_cache, 'wb') as f:
        pickle.dump(xi_all, f)
    print('Sauvegardé.')

# Statistiques sur les courbes ξ
vs_atm_series  = {}  # σ_VS(T) pour T=1M, 1Y, 5Y
for label, T_target in [('1M', 1/12), ('1Y', 1.), ('5Y', 5.)]:
    vals = []
    for d, xi_df in xi_all.items():
        row = xi_df.iloc[(xi_df['T'] - T_target).abs().argsort()].iloc[0]
        vals.append({'date': d, 'vs_vol': np.sqrt(max(row['vs_var'], 0))})
    df_tmp = pd.DataFrame(vals).set_index('date')
    df_tmp.index = pd.to_datetime(df_tmp.index)
    vs_atm_series[label] = df_tmp.sort_index()

print('Extraction terminée.')

In [ ]:
# ============================================================
#  FIGURE 11.1 — σ_VS(T) pour 3 maturités sur 5 ans
# ============================================================
fig, ax = plt.subplots(figsize=(14, 5))

colors_ts = ['steelblue', 'firebrick', 'forestgreen']
for (label, df_vs), col in zip(vs_atm_series.items(), colors_ts):
    ax.plot(df_vs.index, df_vs['vs_vol'] * 100, color=col, lw=1.5, alpha=0.9, label=f'σ_VS({label})')

ax.set_ylabel('VS Volatilité (%)')
ax.set_title('Structure par terme des Variance Swaps — SX5E sur 5 ans')
ax.legend()
ax.xaxis.set_tick_params(rotation=30)
plt.tight_layout()
plt.show()

# Corrélations entre maturités
corr_df = pd.DataFrame({
    '1M': vs_atm_series['1M']['vs_vol'],
    '1Y': vs_atm_series['1Y']['vs_vol'],
    '5Y': vs_atm_series['5Y']['vs_vol'],
}).dropna().corr()

print('\nMatrice de corrélation entre σ_VS(1M), σ_VS(1Y), σ_VS(5Y) :')
print(corr_df.round(3))
print('\n→ Haute corrélation attendue dans le modèle à 2 facteurs.')

In [ ]:
# ============================================================
#  CALIBRATION ω, k1, k2 SUR LES VARIATIONS HISTORIQUES DE ξ
# ============================================================
def compute_vol_of_log_vs(vs_atm_series, horizon_days=21):
    """
    Calcule la vol empirique de ln(σ_VS(τ)) sur horizon=horizon_days jours.
    Retourne {label: vol_annualisée}
    """
    results = {}
    for label, df_vs in vs_atm_series.items():
        log_vs  = np.log(df_vs['vs_vol'].replace(0, np.nan)).dropna()
        delta_log = log_vs.diff(horizon_days).dropna()
        vol_ann = delta_log.std() * np.sqrt(252 / horizon_days)
        results[label] = vol_ann
    return results


volvol_1m_emp  = compute_vol_of_log_vs(vs_atm_series, horizon_days=21)
volvol_12m_emp = compute_vol_of_log_vs(vs_atm_series, horizon_days=252)

print('Vol-de-vol empirique (horizon 1 mois) :')
for k, v in volvol_1m_emp.items():
    model_pred = volvol_2f_dt(OMEGA_REF, K1_REF, K2_REF, THETA_REF, RHO_REF,
                               {'1M': 1/12, '1Y': 1., '5Y': 5.}[k], 1/12)
    print(f'  σ_VS({k}) : empirique={v*100:.1f}%  |  modèle 2F={model_pred*100:.1f}%')

print('\nVol-de-vol empirique (horizon 12 mois) :')
for k, v in volvol_12m_emp.items():
    model_pred = volvol_2f_dt(OMEGA_REF, K1_REF, K2_REF, THETA_REF, RHO_REF,
                               {'1M': 1/12, '1Y': 1., '5Y': 5.}[k], 1.)
    print(f'  σ_VS({k}) : empirique={v*100:.1f}%  |  modèle 2F={model_pred*100:.1f}%')

In [ ]:
# ============================================================
#  CALIBRATION AUTOMATIQUE DE ω, k1, k2
# ============================================================
tau_map = {'1M': 1/12, '1Y': 1., '5Y': 5.}

def objective_calib_2f(params):
    omega, k1, k2 = params
    if omega <= 0 or k1 <= 0 or k2 <= 0 or k1 <= k2:
        return 1e6
    theta = THETA_REF
    rho   = RHO_REF
    loss = 0.
    for label, tau in tau_map.items():
        pred = volvol_2f_dt(omega, k1, k2, theta, rho, tau, dt_y=1/12)
        emp  = volvol_1m_emp[label]
        loss += (pred - emp)**2
    return loss


res_calib = minimize(
    objective_calib_2f, [OMEGA_REF, K1_REF, K2_REF],
    method='Nelder-Mead',
    options={'xatol': 1e-4, 'maxiter': 2000}
)

omega_cal, k1_cal, k2_cal = res_calib.x

print('='*55)
print('  Calibration 2F sur vol-de-vol empirique SX5E')
print('='*55)
print(f'  ω : {omega_cal:.4f}  (Bergomi ref : {OMEGA_REF})')
print(f'  k1: {k1_cal:.4f}  (ref : {K1_REF})  ≈ {12/k1_cal:.1f} mois')
print(f'  k2: {k2_cal:.4f}  (ref : {K2_REF})  ≈ {12/k2_cal:.1f} mois')
print()

# Comparaison
print('Vérification :')
for label, tau in tau_map.items():
    p_cal = volvol_2f_dt(omega_cal, k1_cal, k2_cal, THETA_REF, RHO_REF, tau, 1/12)
    emp   = volvol_1m_emp[label]
    print(f'  σ_VS({label}) : calibré={p_cal*100:.1f}%  |  empirique={emp*100:.1f}%')

In [ ]:
# ============================================================
#  STRUCTURE PAR TERME DU SKEW — CALIBRATION SUR SX5E
# ============================================================
def compute_market_skew(surface, T_target, tol=0.04):
    """Calcule le skew médian σ95%-σ105% depuis la surface."""
    rows = []
    for date, grp in surface.groupby('date'):
        avail = grp['T'].unique()
        clo   = avail[np.argmin(np.abs(avail - T_target))]
        if abs(clo - T_target) > tol:
            continue
        sl = grp[np.abs(grp['T'] - clo) < 1e-3].copy()
        S  = sl['spot'].iloc[0]
        F  = sl['forward'].iloc[0] if 'forward' in sl else S
        # Interpolation au niveau 95% et 105%
        sl_s = sl.sort_values('strike')
        if len(sl_s) < 4:
            continue
        try:
            iv_interp = interp1d(sl_s['strike'].values, sl_s['market_iv'].values,
                                 kind='linear', fill_value='extrapolate')
            iv95  = float(iv_interp(F * 0.95))
            iv105 = float(iv_interp(F * 1.05))
            rows.append({'date': date, 'skew_95_105': iv95 - iv105, 'T': clo})
        except Exception:
            pass
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    return df.set_index('date').sort_index()


from scipy.interpolate import interp1d as interp1d  # assurer l'import

target_mats = {'1M': 1/12, '3M': 3/12, '6M': 6/12, '1Y': 1., '2Y': 2.}
skew_market = {}
for label, T in target_mats.items():
    skew_market[label] = compute_market_skew(surface, T, tol=0.05)

# Skew médian par maturité
skew_medians = {}
for label, df_sk in skew_market.items():
    if len(df_sk) > 0 and 'skew_95_105' in df_sk:
        skew_medians[label] = df_sk['skew_95_105'].median()

print('Skew 95%–105% médian SX5E sur 5 ans :')
mat_months_mkt = []
sk_mkt         = []
for label, T in target_mats.items():
    if label in skew_medians:
        val = skew_medians[label]
        mat_months_mkt.append(T * 12)
        sk_mkt.append(val)
        print(f'  {label} : {val*100:.3f}%')

In [ ]:
# ============================================================
#  CALIBRATION ρ_SX, ρ_SY SUR LE SKEW OBSERVÉ
# ============================================================
# On calibre le skew 1M (donne Skew∆) et le ratio long/court (donne ρ_SX, ρ_SY)

if len(sk_mkt) >= 2 and '1M' in skew_medians:
    Skew_1M_mkt = skew_medians['1M']

    def objective_rho(params):
        rSX, rSY = params
        loss = 0.
        for label, T in target_mats.items():
            if label not in skew_medians:
                continue
            N = max(1, int(T * 12))
            mats_th, sk_th, _, _ = skew_term_structure(
                Skew_1M_mkt, omega_cal, k1_cal, k2_cal,
                THETA_REF, rSX, rSY, N_max=N, delta=1/12
            )
            sk_pred = sk_th[-1]
            sk_mkt_val = skew_medians[label]
            loss += (sk_pred - sk_mkt_val)**2
        return loss

    res_rho = minimize(objective_rho, [RHO_SX_REF, RHO_SY_REF],
                       method='Nelder-Mead',
                       options={'xatol': 1e-4, 'maxiter': 1000})
    rSX_cal, rSY_cal = res_rho.x
    rSX_cal = np.clip(rSX_cal, -0.99, -0.01)
    rSY_cal = np.clip(rSY_cal, -0.99, 0.99)

    print(f'Calibration ρ_SX, ρ_SY :')
    print(f'  ρ_SX calibré : {rSX_cal:.3f}  (ref : {RHO_SX_REF})')
    print(f'  ρ_SY calibré : {rSY_cal:.3f}  (ref : {RHO_SY_REF})')

    # Graphique de la structure par terme calibrée vs marché
    mats_model, sk_model, sk_intr_m, sk_sv_m = skew_term_structure(
        Skew_1M_mkt, omega_cal, k1_cal, k2_cal,
        THETA_REF, rSX_cal, rSY_cal, N_max=60
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(mats_model, sk_model * 100, 'steelblue', lw=2.5, label='Modèle calibré')
    if mat_months_mkt:
        ax.scatter(mat_months_mkt, np.array(sk_mkt) * 100, color='firebrick',
                   s=80, zorder=5, label='Marché (médiane 5 ans)')
    ax.set_xlabel('Maturité (mois)')
    ax.set_ylabel('Skew 95%–105% (%)')
    ax.set_title('Structure par terme du skew : modèle calibré vs SX5E')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    rSX_cal, rSY_cal = RHO_SX_REF, RHO_SY_REF
    print('Données insuffisantes — utilisation des paramètres de référence.')

---
## Section 12 — Synthèse : Bergomi I vs Bergomi II

### 12.1 Comparaison structurelle

| Propriété | Bergomi I (Heston) | Bergomi II (FV) |
|---|---|---|
| Variables d'état | $(S, V)$ | $(S, X, Y)$ ou $(S, \xi_1, \dots, \xi_N)$ |
| Forward skew | Contraint par $\sigma$ et $\rho$ | **Contrôlé séparément** (CEV) |
| Vol-de-vol structure/terme | Une seule échelle ($\sigma$) | **Deux échelles** ($k_1$, $k_2$) |
| Corrélation spot/vol | Un seul paramètre ($\rho$) | **Deux paramètres** ($\rho_{SX}$, $\rho_{SY}$) |
| Calibration VS | Non explicite | **Calibration exacte** par construction |
| Options sur variance | Non cohérent | **Pricées cohéremment** |
| Limitation structurelle | $\sigma_{réal} / \sigma_{impl} \approx 0.6$ | Pas de contrainte rigide |

### 12.2 Résultats empiriques SX5E

In [ ]:
# ============================================================
#  DASHBOARD FINAL — BERGOMI II SX5E
# ============================================================
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Structure par terme σ_VS sur date récente
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(vs_curve['T'] * 12, vs_curve['vs_vol'] * 100, 'o-', color='steelblue', lw=2, ms=5)
ax1.set_title(f'σ_VS(T) — {latest_date.date()}')
ax1.set_xlabel('T (mois)')
ax1.set_ylabel('σ_VS (%)')

# 2. σ_VS(1M) sur 5 ans
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(vs_atm_series['1M'].index, vs_atm_series['1M']['vs_vol'] * 100,
         'steelblue', lw=1.2)
ax2.set_title('σ_VS(1M) — 5 ans')
ax2.xaxis.set_tick_params(rotation=30, labelsize=7)

# 3. Vol-de-vol : 2F vs NF
ax3 = fig.add_subplot(gs[0, 2])
taus_plot = np.array([1, 3, 6, 12, 24, 36, 48, 60])
vv_2f_p = [volvol_2f_dt(omega_cal, k1_cal, k2_cal, THETA_REF, RHO_REF, t/12, 1/12)*100
            for t in taus_plot]
ax3.plot(taus_plot, vv_2f_p, 'o-', color='steelblue', lw=2, ms=5, label='2F calibré')
ax3.plot(taus_months, [v*100 for v in volvol_1m_emp.values()], 'rs', ms=8, label='Empirique')
ax3.set_title('Vol-de-Vol (Δt=1M)')
ax3.set_xlabel('τ (mois)')
ax3.legend(fontsize=8)

# 4. Structure par terme du skew
ax4 = fig.add_subplot(gs[1, 0])
mats_th2, sk_th2, sk_intr2, sk_sv2 = skew_term_structure(
    skew_medians.get('1M', 0.05), omega_cal, k1_cal, k2_cal,
    THETA_REF, rSX_cal, rSY_cal, N_max=60
)
ax4.plot(mats_th2, sk_th2 * 100, 'steelblue', lw=2)
ax4.plot(mat_months_mkt, np.array(sk_mkt) * 100, 'rs', ms=8)
ax4.set_title('Skew 95%–105%')
ax4.set_xlabel('T (mois)')

# 5. Décomposition skew
ax5 = fig.add_subplot(gs[1, 1])
ax5.fill_between(mats_th2, sk_intr2*100, alpha=0.6, color='steelblue', label='Intrinsèque')
ax5.fill_between(mats_th2, sk_intr2*100, sk_th2*100, alpha=0.6, color='firebrick', label='Spot/Vol')
ax5.set_title('Décomposition du skew')
ax5.set_xlabel('T (mois)')
ax5.legend(fontsize=8)

# 6. Corrélation entre maturités VS
ax6 = fig.add_subplot(gs[1, 2])
jdf = pd.DataFrame({
    '1M': vs_atm_series['1M']['vs_vol'],
    '1Y': vs_atm_series['1Y']['vs_vol'],
}).dropna()
ax6.scatter(jdf['1M']*100, jdf['1Y']*100, alpha=0.2, s=5, color='steelblue')
ax6.set_xlabel('σ_VS(1M) (%)')
ax6.set_ylabel('σ_VS(1Y) (%)')
ax6.set_title(f'σ_VS(1M) vs σ_VS(1Y) — corr={jdf.corr().iloc[0,1]:.3f}')

# 7. Prix exotiques : BS vs Full
ax7 = fig.add_subplot(gs[2, 0])
prods = ['Reverse Cliquet', 'Napoleon', 'Accumulator']
x_pos = np.arange(len(prods))
bs_vals   = [results_table['Black-Scholes'][p]*100 for p in prods]
full_vals = [results_table['Full (Bergomi)'][p]*100 for p in prods]
w = 0.35
ax7.bar(x_pos - w/2, bs_vals,   w, color='#4e79a7', label='BS')
ax7.bar(x_pos + w/2, full_vals, w, color='#e15759', label='Full')
ax7.set_xticks(x_pos)
ax7.set_xticklabels(['RC', 'Nap', 'Acc'], fontsize=9)
ax7.set_title('BS vs Full Bergomi')
ax7.set_ylabel('Prix (%)')
ax7.legend(fontsize=8)

# 8. Paramètres calibrés — tableau
ax8 = fig.add_subplot(gs[2, 1])
ax8.axis('off')
params_data = [
    ['Paramètre', 'Bergomi ref', 'SX5E calibré'],
    ['ω',    f'{OMEGA_REF:.3f}', f'{omega_cal:.3f}'],
    ['k₁',   f'{K1_REF:.1f}',   f'{k1_cal:.3f}'],
    ['k₂',   f'{K2_REF:.2f}',   f'{k2_cal:.3f}'],
    ['θ',    f'{THETA_REF:.2f}', 'fixé'],
    ['ρ_SX', f'{RHO_SX_REF:.2f}', f'{rSX_cal:.3f}'],
    ['ρ_SY', f'{RHO_SY_REF:.3f}', f'{rSY_cal:.3f}'],
]
table = ax8.table(cellText=params_data[1:], colLabels=params_data[0],
                   cellLoc='center', loc='center',
                   colColours=['#deeaf1']*3)
table.auto_set_font_size(False)
table.set_fontsize(9)
ax8.set_title('Paramètres calibrés', pad=20)

# 9. Distribution ξ^T(1M)
ax9 = fig.add_subplot(gs[2, 2])
xi_1m_vals = []
for d, xi_df in xi_all.items():
    row = xi_df.iloc[(xi_df['T'] - 1/12).abs().argsort()].iloc[0]
    if row['xi'] > 0:
        xi_1m_vals.append(np.sqrt(row['xi']))
ax9.hist(np.array(xi_1m_vals) * 100, bins=60, density=True,
         color='steelblue', alpha=0.7)
ax9.set_xlabel('√ξ^(1M) = σ_VS(1M) (%)')
ax9.set_title('Distribution de σ_VS(1M) — 5 ans')

plt.suptitle('Dashboard Bergomi II — SX5E, 5 ans glissants',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('bergomi2_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard sauvegardé : bergomi2_dashboard.png')

In [ ]:
# ============================================================
#  BILAN QUANTITATIF FINAL
# ============================================================
print('=' * 72)
print('  BILAN — Smile Dynamics II (Bergomi 2005) sur SX5E, 5 ans')
print('=' * 72)
print(f'''
  DONNÉES
  ─────────────────────────────────────────────────────────────────
  Période      : {date_start.date()}  →  {date_end.date()}
  Dates VS     : {len(xi_all)}
  σ_VS(1M) médiane  : {vs_atm_series["1M"]["vs_vol"].median()*100:.2f}%
  σ_VS(1Y) médiane  : {vs_atm_series["1Y"]["vs_vol"].median()*100:.2f}%

  MODÈLE À 2 FACTEURS — PARAMÈTRES CALIBRÉS SUR SX5E
  ─────────────────────────────────────────────────────────────────
  ω  = {omega_cal:.4f}  (ref Bergomi : {OMEGA_REF})
  k₁ = {k1_cal:.4f}  (ref : {K1_REF})  → τ₁ ≈ {12/k1_cal:.1f} mois
  k₂ = {k2_cal:.4f}  (ref : {K2_REF})  → τ₂ ≈ {12/k2_cal:.1f} mois
  θ  = {THETA_REF} (fixé)
  ρ_SX = {rSX_cal:.3f}
  ρ_SY = {rSY_cal:.3f}

  VOL-DE-VOL (horizon 1 mois, calibré)
  ─────────────────────────────────────────────────────────────────
  σ_VS(1M)  : {volvol_2f_dt(omega_cal, k1_cal, k2_cal, THETA_REF, RHO_REF, 1/12, 1/12)*100:.1f}%
  σ_VS(1Y)  : {volvol_2f_dt(omega_cal, k1_cal, k2_cal, THETA_REF, RHO_REF, 1., 1/12)*100:.1f}%
  σ_VS(5Y)  : {volvol_2f_dt(omega_cal, k1_cal, k2_cal, THETA_REF, RHO_REF, 5., 1/12)*100:.1f}%

  PRIX DES EXOTIQUES (MC, N=20000 chemins, Δ=1 mois)
  ─────────────────────────────────────────────────────────────────
  Produit          | Black-Scholes | Full Bergomi | Ratio
  Reverse Cliquet  | {results_table["Black-Scholes"]["Reverse Cliquet"]*100:.2f}%         | {results_table["Full (Bergomi)"]["Reverse Cliquet"]*100:.2f}%        | {results_table["Full (Bergomi)"]["Reverse Cliquet"]/results_table["Black-Scholes"]["Reverse Cliquet"]:.1f}x
  Napoleon         | {results_table["Black-Scholes"]["Napoleon"]*100:.2f}%         | {results_table["Full (Bergomi)"]["Napoleon"]*100:.2f}%        | {results_table["Full (Bergomi)"]["Napoleon"]/results_table["Black-Scholes"]["Napoleon"]:.1f}x
  Accumulator      | {results_table["Black-Scholes"]["Accumulator"]*100:.2f}%         | {results_table["Full (Bergomi)"]["Accumulator"]*100:.2f}%        | {results_table["Full (Bergomi)"]["Accumulator"]/results_table["Black-Scholes"]["Accumulator"]:.1f}x

  CONTRIBUTIONS AU PRIX (vs Black-Scholes)
  ─────────────────────────────────────────────────────────────────
  Reverse Cliquet : Skew={( results_table["Forward Skew uniquement"]["Reverse Cliquet"]-results_table["Black-Scholes"]["Reverse Cliquet"])*100:.2f}%  |  VoV={( results_table["Vol-de-Vol uniquement"]["Reverse Cliquet"]-results_table["Black-Scholes"]["Reverse Cliquet"])*100:.2f}%
  Napoleon        : Skew={( results_table["Forward Skew uniquement"]["Napoleon"]-results_table["Black-Scholes"]["Napoleon"])*100:.2f}%  |  VoV={( results_table["Vol-de-Vol uniquement"]["Napoleon"]-results_table["Black-Scholes"]["Napoleon"])*100:.2f}%
  Accumulator     : Skew={( results_table["Forward Skew uniquement"]["Accumulator"]-results_table["Black-Scholes"]["Accumulator"])*100:.2f}%  |  VoV={( results_table["Vol-de-Vol uniquement"]["Accumulator"]-results_table["Black-Scholes"]["Accumulator"])*100:.2f}%
''')
print('=' * 72)

---
## Annexe — Paramétrisation de χ et décorrélation du skew forward

### A.1 Paramétrisation de ρ_SY via χ

Bergomi utilise la paramétrisation suivante pour garantir que $|\rho_{SY}|$ soit physiquement cohérent avec $\rho_{SX}$ et $\rho_{UV}$ :

$$\rho_{SY} = \rho_{SX}\,\rho_{UV} + \chi\sqrt{1 - \rho_{SX}^2}\sqrt{1 - \rho_{UV}^2}, \qquad \chi \in [-1, 1]$$

### A.2 Propriété clé : découplage skew/corrélation

**Dans les modèles stochastiques classiques** (Heston), changer $\rho$ change **simultanément** le skew forward et la corrélation spot/vol.

**Dans le modèle Bergomi II** : grâce à la spécification CEV discrète, le skew mensuel ($\Delta = 1\text{ mois}$) est **indépendant** de $\rho_{SX}$ et $\rho_{SY}$. Ces paramètres contrôlent uniquement le **skew de terme** (via la contribution spot/vol dans l'équation 4.4).

In [ ]:
# ============================================================
#  ILLUSTRATION DU DÉCOUPLAGE — Skew 1M indépendant de ρ_SX
# ============================================================
rho_SX_vals = [-0.90, -0.70, -0.50, -0.30, 0.0]
mats_fine, _, _, _ = skew_term_structure(
    SKEW_DELTA, OMEGA_REF, K1_REF, K2_REF, THETA_REF, -0.70, -0.357
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Skew total pour différents ρ_SX
for rSX in rho_SX_vals:
    m, sk, _, _ = skew_term_structure(
        SKEW_DELTA, OMEGA_REF, K1_REF, K2_REF, THETA_REF, rSX, rSX * 0.51
    )
    axes[0].plot(m, sk * 100, lw=2, label=f'ρ_SX = {rSX:.0%}')
axes[0].axvline(1, ls=':', color='gray', lw=1, label='T=1M (découplé)')
axes[0].set_xlabel('Maturité (mois)')
axes[0].set_ylabel('Skew 95%–105% (%)')
axes[0].set_title('Le skew 1M est identique pour tous les ρ_SX\n(propriété de découplage)')
axes[0].legend(fontsize=9)

# Contribution spot/vol à différents horizons
N_vals = [1, 3, 6, 12, 24, 36, 60]
for rSX in [-0.90, -0.70, -0.50, 0.0]:
    sv_contribs = []
    for N in N_vals:
        m, _, _, sv = skew_term_structure(
            SKEW_DELTA, OMEGA_REF, K1_REF, K2_REF, THETA_REF,
            rSX, rSX * 0.51, N_max=N, delta=1/12
        )
        sv_contribs.append(sv[-1])
    axes[1].plot(N_vals, np.array(sv_contribs) * 100, 'o-', lw=2,
                 label=f'ρ_SX = {rSX:.0%}')
axes[1].set_xlabel('Maturité (mois)')
axes[1].set_ylabel('Contribution spot/vol au skew (%)')
axes[1].set_title('Contribution corrélation spot/vol')
axes[1].legend(fontsize=9)

plt.suptitle('Annexe — Propriété de découplage skew forward / corrélation spot/vol',
             fontweight='bold')
plt.tight_layout()
plt.show()

---

## Références

- **Bergomi, L. (2005)** — *Smile Dynamics II*, Société Générale (SSRN 1493302)
- **Bergomi, L. (2004)** — *Smile Dynamics I*, Risk September 2004
- **Dupire, B. (1996)** — *A unified theory of volatility*, unpublished
- **Carr, P., Geman, H., Madan, D., Yor, M. (2003)** — *Stochastic Volatility for Lévy Processes*, Mathematical Finance
- **Backus, D., Foresi, S., Li, K., Wu, L. (1997)** — *Accounting for biases in Black-Scholes*, unpublished
- **Leland, H. (1985)** — *Option replication with transaction costs*, Journal of Finance
- **Zhou, F. (2003)** — *Black smirks*, Risk January

---

## Résumé des équations clés

| Équation | Formule | Référence papier |
|---|---|---|
| Forward Variance (1F) | $\xi^T(t) = \xi^T(0)\exp(\omega e^{-k_1\tau}X_t - \frac{\omega^2}{2}e^{-2k_1\tau}\mathbb{E}[X_t^2])$ | Éq. (2.1) |
| Forward Variance (2F) | $\xi^T(t) = \xi^T(0)\exp(\omega[e^{-k_1\tau}X + \theta e^{-k_2\tau}Y] - \frac{\omega^2}{2}[\cdot])$ | Éq. (2.2) |
| Corrélation N-facteurs | $\rho(Z^i, Z^j) = \theta\rho_0 + (1-\theta)\beta^{|j-i|}$ | Éq. (2.4) |
| Processus spot CEV | $dS = (r-q)S\,dt + \sigma_0(S/S_{T_i})^{1-\beta}S\,dZ$ | Éq. (3.1) |
| Skew terme (analytique) | $\text{Skew}_{N\Delta} = \text{Skew}_\Delta/N + \omega\sqrt{2}[\rho_{SX}\zeta(k_1\Delta,N) + \theta\rho_{SY}\zeta(k_2\Delta,N)]$ | Éq. (4.4) |
| Fonction ζ | $\zeta(x,N) = \frac{1-e^{-x}}{x}\cdot\frac{\sum_{\tau=1}^{N-1}(N-\tau)e^{-(\tau-1)x}}{N^2}$ | Éq. (4.3) |

---
*Notebook complet — Bergomi (2005) Smile Dynamics II sur SX5E*